# Pocket Agent
> **Build a fully functional local coding agent from scratch.**
> 15 chapters · one notebook · no API costs.

Each chapter adds exactly one capability.
Run a chapter's cells top-to-bottom and you get a working agent at the end of that chapter.

| Ch | Title | What you gain |
|----|-------|---------------|
| 1  | Setup | Install Ollama, pull a model, configure the notebook |
| 2  | Hello Ollama | Talk to a local LLM from Python |
| 3  | Context is a Budget | Understand why context management exists |
| 4  | Give it a Map | Load AGENTS.md as an advisory project manifest |
| 5  | Glob + JIT Reads | Navigate a file tree without bulk-loading files |
| 6  | Fuzzy Scoring | Rank retrieved files, not just find them |
| 7  | Grep | Find code by content, not just filename |
| 8  | Microcompaction | Hot/cold storage — survive long sessions |
| 9  | Semantic RAG | Retrieval that understands meaning, not just keywords |
| 10 | Full Pipeline | One `run(query, repo)` call does everything |
| 11 | Write + Diff | The agent modifies files on disk |
| 12 | Agent Loop | Autonomous read → plan → write → verify loop |
| 13 | Test Generation | Agent writes and verifies its own tests |
| 14 | Adding a Capability | The three-step pattern for extending the agent |
| 15 | Streamlit App | A browser UI that uses everything you built |

---


---
## Chapter 1 — Setup

Get Ollama installed, pull a model, and configure the notebook.

| Step | Type | What it does |
|------|------|-------------|
| 1.1 | read | Install Ollama — follow the instructions |
| 1.2 | run  | Pull your chosen model |
| 1.3 | run  | Auto-configure token budget and settings |

### 1.1 Install Ollama

Download and install Ollama for your platform from **[ollama.com/download](https://ollama.com/download)**.

Once installed, start the Ollama daemon if it isn't already running:

```bash
ollama serve
```

> On macOS and Windows, Ollama starts automatically when you open the app.  
> On Linux, run `ollama serve` in a separate terminal and leave it open.

Browse available models at **[ollama.com/library](https://ollama.com/library)**.  
Cloud models (tagged `cloud`) run on Ollama's servers — no GPU or download required, just `ollama signin` first.

### 1.2 Pull a model

Set `MODEL_TO_PULL` below to whichever model you want, then run the cell.  
It calls `ollama pull` for you — no terminal needed. We use `qwen3-coder-next` 
for demonstration purposes, but you can use any model (preferably a coding model). 

**Recommendations:**

| Model | Size |
|-------|------|
| `qwen3-coder-next:cloud` | - (cloud) |
| `qwen3-coder-next` | ~52 GB (local) | 

For embeddings (Chapter 9) `nomic-embed-text` is always pulled alongside your chat model.

**The nomic-embed-text model**

Unlike chat models, `nomic-embed-text` doesn't generate text — it converts a piece of text into a 
list of numbers (a vector) that represents its meaning. Two pieces of text that mean similar things 
will produce similar vectors. Chapter 9 uses this to find files that are *semantically* related to 
your query, even if they share no keywords in common. It's small (~274 MB) because it only needs to 
encode meaning, not generate language.

In [11]:
import subprocess, sys

# ── Change this to the model you want ────────────────────────────────────────
MODEL_TO_PULL = "qwen3-coder-next:cloud"   # ← edit me

def _pull(model: str) -> None:
    """Pull an Ollama model, suppressing the spinner output that clutters notebooks."""
    print(f"Pulling {model} ...", end=" ", flush=True)
    result = subprocess.run(
        ["ollama", "pull", model],
        stdout=subprocess.DEVNULL,   # progress bars go to stdout
        stderr=subprocess.DEVNULL,   # spinner frames go to stderr
    )
    if result.returncode == 0:
        print("done ✓")
    else:
        print("FAILED ✗")
        raise RuntimeError(f"ollama pull {model} exited with code {result.returncode}")

_pull(MODEL_TO_PULL)
_pull("nomic-embed-text")   # needed for Chapter 9 semantic search

print("\nAll models ready. Proceed to cell 1.3.")


Pulling qwen3-coder-next:cloud ... done ✓
Pulling nomic-embed-text ... done ✓

All models ready. Proceed to cell 1.3.


### 1.3 Run configuration

Sets up all the variables the notebook uses throughout every chapter. The token budget is 
auto-detected from the model you pulled in 1.2 — you don't need to change anything here 
unless you want to point the agent at a different folder (`REPO_ROOT`) or swap models later.

## Global Configuration

In [12]:
# ── Known context windows — add your model here if it's missing ──────────────
_CONTEXT_WINDOWS = {
    "qwen3-coder-next": 262144,
    "qwen3-coder":      131072,
    "qwen4.5":           32768,
    "qwen3":             32768,
    "llama4.2":          32768,
    "llama4.1":          32768,
    "mistral":           32768,
    "codellama":          4096,
    "gemma3":            32768,
    "devstral-small-2":  32768,
}

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = MODEL_TO_PULL                    # set in cell 1.2
OLLAMA_EMBED    = "nomic-embed-text"
REPO_ROOT       = "."
USE_WEB_UI      = False
REBUILD_INDEX   = False

# Auto-detect token budget from model name prefix
_base        = OLLAMA_MODEL.split(":")[0]          # strip tag  e.g. "qwen4.5:4b" → "qwen4.5"
if _base not in _CONTEXT_WINDOWS:
    print(f"WARNING: '{_base}' not in _CONTEXT_WINDOWS — defaulting to 32,768 tokens.")
    print(f"  Add it to _CONTEXT_WINDOWS above if you know the correct context length.")
    print(f"  Run: ollama show {OLLAMA_MODEL}  and look for 'context length'")
TOKEN_BUDGET = _CONTEXT_WINDOWS.get(_base, 32768)  # default 32K if model not in table

print(f"Model        : {OLLAMA_MODEL}")
print(f"Embed model  : {OLLAMA_EMBED}")
print(f"Token budget : {TOKEN_BUDGET:,}")
print(f"Repo root    : {REPO_ROOT}")

Model        : qwen3-coder-next:cloud
Embed model  : nomic-embed-text
Token budget : 262,144
Repo root    : .


---
## Chapter 2 — Hello Ollama

**Goal:** create the two things everything else builds on —
a function that talks to Ollama, and a status panel that exposes the
agent's internal state at every step.

**You will:**
- Verify Ollama is reachable
- Write `chat()` — the single function every later chapter calls
- Write `show_panel()` — an HTML panel showing the token budget, retrieved files,
  active strategy, and assembled prompt size — rendered natively in the notebook
- Run a test query end-to-end

> **Token budget** is how much of the model's context window your current prompt is using.
> Every model has a fixed limit (e.g. 32,768 tokens). As the agent loads more files into context,
> the budget fills up. When it's full, no more files can be added — the model simply won't see them.
> `show_panel()` makes this visible so you always know how much room you have left.


### 2.1 Dependencies

Chapter 2 only needs `requests` (HTTP to Ollama).
`IPython.display` is part of Jupyter itself — no install needed.
Later chapters will `pip install` their own additions at the top of their section.


In [13]:
# Ensure Chapter 2 dependencies are installed in the current Python environment.
import subprocess, sys

_CH2_DEPS = ["requests"]   # requests: HTTP calls to Ollama
                            # IPython.display is bundled with Jupyter — no install needed

for _pkg in _CH2_DEPS:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        stdout=subprocess.DEVNULL,
    )

print("Chapter 2 dependencies ready:", _CH2_DEPS)


Chapter 2 dependencies ready: ['requests']


### 2.2 Checking Ollama is running

Before sending any messages we probe the `/api/tags` endpoint.
That endpoint returns the list of locally pulled models — a useful
sanity check that both the server and the model we want are present.

In [14]:
import requests

def ping_ollama() -> tuple[bool, list[str] | str]:
    """
    Return (ok, info).
    ok=True  → info is a list of pulled model names.
    ok=False → info is the error message.
    """
    try:
        # GET /api/tags returns metadata for every model Ollama has pulled locally
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()                              # throw an error on 4xx / 5xx
        models = [m["name"] for m in r.json().get("models", [])]
        return True, models
    except Exception as exc:
        return False, str(exc)                            # return the error as-is

ok, info = ping_ollama()

if ok:
    print(f"Ollama is running.")
    print(f"Pulled models: {info}")

    # Soft warning — the rest of the notebook will fail if the model isn't present,
    # but we don't hard-crash here so the user can read the message clearly
    if OLLAMA_MODEL not in " ".join(info):
        print(f"\n  WARNING: '{OLLAMA_MODEL}' not found in pulled models.")
        print(f"  Run: ollama pull {OLLAMA_MODEL}")
else:
    print(f"Cannot reach Ollama at {OLLAMA_BASE_URL}")
    print(f"Error: {info}")
    print("Start it with: ollama serve")     # most common fix — Ollama daemon not running

Ollama is running.
Pulled models: ['nomic-embed-text:latest', 'qwen3-coder-next:cloud', 'qwen3.5:4b', 'deepseek-v3.1:671b-cloud', 'qwen3-coder:480b-cloud', 'codellama:7b']


### 2.3 The `chat()` function

`chat()` is the only function that ever calls Ollama.
Every chapter — from the simplest single-turn query to the full
autonomous agent loop — goes through this one function.

It takes a standard OpenAI-style `messages` list and returns
the reply text plus the token count Ollama reports.
The token count is what populates the budget bar in the status panel.

In [16]:
def chat(
    messages: list[dict],
    model: str = OLLAMA_MODEL,
) -> tuple[str, int]:
    """
    Send *messages* to Ollama, return (reply_text, tokens_used).
    tokens_used is the sum of prompt and completion tokens as reported by Ollama.
    """
    payload = {
        "model":    model,
        "messages": messages,
        "stream":   False,
        "options":  {"num_ctx": TOKEN_BUDGET},
    }
    r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=120)
    r.raise_for_status()
    data   = r.json()
    reply  = data["message"]["content"]
    tokens = data.get("prompt_eval_count", 0) + data.get("eval_count", 0)
    return reply, tokens


def chat_continued(
    messages:          list[dict],
    model:             str = OLLAMA_MODEL,
) -> tuple[str, int]:
    """
    Like chat(), but automatically continues if the model stops mid-response.

    Ollama sets done_reason="length" when it hits the token limit instead of
    finishing naturally ("stop").  When that happens we append the partial reply
    as an assistant turn, add a "Continue." user turn, and call again — exactly
    the same mechanism Claude's API uses internally.

    max_continuations caps the loop so a runaway model can't spin forever.
    """
    full_reply   = ""
    total_tokens = 0
    msgs         = list(messages)

    while True:
        payload = {
            "model":    model,
            "messages": msgs,
            "stream":   False,
            "options":  {"num_ctx": TOKEN_BUDGET},
        }
        r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=180)
        r.raise_for_status()
        data = r.json()

        chunk        = data["message"]["content"]
        full_reply  += chunk
        total_tokens += data.get("prompt_eval_count", 0) + data.get("eval_count", 0)

        # "stop"   → model finished naturally
        # "length" → hit the token ceiling mid-response — continue
        if data.get("done_reason", "stop") != "length":
            break

        if turn < max_continuations:
            # Append partial reply + continuation prompt and loop
            msgs = msgs + [
                {"role": "assistant", "content": chunk},
                {"role": "user",      "content": "Continue."},
            ]

    return full_reply, total_tokens


### 2.4 The status panel

The panel is the same in every chapter — only the data it receives changes.
It shows four things:

| Row | What it shows |
|-----|--------------|
| **Token Budget** | A coloured progress bar. Green < 60 %, yellow < 85 %, red above. |
| **Retrieved** | Every file the agent loaded, tagged HOT (in prompt) or COLD (offloaded). |
| **Strategy** | Which retrieval strategy was used: glob, fuzzy, grep, or semantic. |
| **Prompt size** | The assembled prompt in tokens, *before* the LLM call. |

`show_panel()` is intentionally stateless — you call it with the current
snapshot before the LLM call and again after, so you can see both.

In [18]:
from IPython.display import HTML, Markdown, display


def show_rule(title: str = "") -> None:
    """Render a horizontal rule with a centred title — native notebook HTML."""
    safe = title.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML(
        f'<div style="display:flex;align-items:center;margin:14px 0 6px;gap:8px;font-family:monospace">'
        f'<hr style="flex:1;border:none;border-top:1px solid #d0d7de;margin:0">'
        f'<b style="color:#57606a;white-space:nowrap;font-size:0.9rem">{safe}</b>'
        f'<hr style="flex:1;border:none;border-top:1px solid #d0d7de;margin:0">'
        f'</div>'
    ))


def show_panel(
    query:        str,
    token_used:   int,
    token_budget: int        = TOKEN_BUDGET,
    files:        list[dict] | None = None,
    strategy:     str        = "none",
    prompt_size:  int        = 0,
) -> None:
    """
    Render the Pocket Agent status panel as HTML in the notebook output.

    Parameters
    ----------
    query        : the user's query (shown in the panel title)
    token_used   : tokens consumed so far
    token_budget : total context-window size (default TOKEN_BUDGET)
    files        : list of dicts {"path": str, "hot": bool}
    strategy     : retrieval strategy name
    prompt_size  : estimated assembled-prompt size in tokens
    """
    files = files or []
    pct   = token_used / token_budget if token_budget > 0 else 0
    bar_w = min(int(pct * 200), 200)           # px, capped at full bar
    bar_color = (
        "#2da44e" if pct < 0.70 else
        "#bf8700" if pct < 0.85 else
        "#cf222e"
    )

    # ── file badges ────────────────────────────────────────────────────────────
    file_rows = ""
    for f in files:
        if f["hot"]:
            badge = '<span style="background:#cf222e;color:#fff;border-radius:3px;padding:1px 6px;font-size:0.78rem;font-weight:bold">HOT</span>'
        else:
            badge = '<span style="background:#0969da;color:#fff;border-radius:3px;padding:1px 6px;font-size:0.78rem;font-weight:bold">COLD</span>'
        path  = f["path"].replace("&","&amp;").replace("<","&lt;")
        file_rows += f'<div style="margin:2px 0">{badge}&nbsp;&nbsp;<code style="font-size:0.85rem">{path}</code></div>'

    if not file_rows:
        file_rows = '<span style="color:#8c959f;font-style:italic">none</span>'

    short_q = query[:80] + ("…" if len(query) > 80 else "")
    short_q = short_q.replace("&","&amp;").replace("<","&lt;")

    html = f"""
<div style="border:1px solid #d0d7de;border-radius:8px;padding:14px 18px;
            margin:8px 0;background:#f6f8fa;font-family:monospace;font-size:0.88rem">
  <div style="font-weight:bold;color:#0969da;margin-bottom:10px">
    Pocket Agent&nbsp;&nbsp;·&nbsp;&nbsp;<em style="font-weight:normal;color:#57606a">{short_q}</em>
  </div>
  <table style="border-collapse:collapse;width:100%">
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a;vertical-align:middle;white-space:nowrap">Token Budget</td>
      <td style="vertical-align:middle">
        <div style="display:inline-flex;align-items:center;gap:10px">
          <div style="width:200px;height:10px;border-radius:5px;background:#e0e0e0;overflow:hidden">
            <div style="width:{bar_w}px;height:100%;background:{bar_color};transition:width 0.3s"></div>
          </div>
          <span style="color:#57606a;font-size:0.82rem">{token_used:,}&nbsp;/&nbsp;{token_budget:,} tokens</span>
        </div>
      </td>
    </tr>
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a;vertical-align:top">Retrieved</td>
      <td style="vertical-align:top">{file_rows}</td>
    </tr>
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a">Strategy</td>
      <td style="color:#8250df;font-weight:bold">{strategy}</td>
    </tr>
    <tr>
      <td style="padding:4px 14px 4px 0;color:#57606a">Prompt&nbsp;size</td>
      <td style="color:#57606a">{prompt_size:,} tokens</td>
    </tr>
  </table>
</div>"""
    display(HTML(html))


def show_reply(text: str) -> None:
    """
    Render the model's reply as markdown in the notebook output.
    Falls back to plain print() outside of Jupyter.
    """
    try:
        display(Markdown(text))
    except Exception:
        print(text)


### 2.5 Try it

Send a single question to Ollama and observe the panel
before the call (prompt assembled, no reply yet) and after
(real token count from Ollama). The response will be rendered as markdown just below the panel.

The token count before the call is a rough estimate (`len(text) // 4`).
From Chapter 3 onward we'll use a proper tokeniser.

In [20]:
query    = "What is a context window, and why does its size matter?"
messages = [{"role": "user", "content": query}]

# Rough pre-call estimate: 1 token ≈ 4 characters
prompt_size = len(query) // 4

show_rule("Before LLM call")
show_panel(
    query        = query,
    token_used   = prompt_size,
    strategy     = "none",
    prompt_size  = prompt_size,
)

print("\nSending query to Ollama…\n")
reply, tokens_used = chat_continued(messages)

show_rule("After LLM call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    strategy    = "none",
    prompt_size = prompt_size,
)

show_reply(reply)


Token Budget,"13 / 262,144 tokens"
Retrieved,none
Strategy,none
Prompt size,13 tokens



Sending query to Ollama…



Token Budget,"510 / 262,144 tokens"
Retrieved,none
Strategy,none
Prompt size,13 tokens


A **context window** is the maximum amount of text (in tokens—subword units like words or subwords) that a language model can process and consider *at once* during inference or training. It defines how much historical information (e.g., previous turns in a conversation, prior sentences in a document) the model can “see” and use to generate a response.

### Why size matters:
1. **Coherence & Long-Dependence Understanding**  
   Larger context windows allow the model to maintain context over longer conversations or documents—e.g., remembering a character introduced in chapter 1 of a story, or tracking legal clauses across hundreds of pages. Small windows may cause the model to “forget” earlier parts, leading to inconsistencies.

2. **Handling Long-Form Tasks**  
   Tasks like summarizing a 10,000-word report, answering detailed questions about a lengthy PDF, or coding with large codebases require large context windows. With a small context, you must chunk and stitch responses, risking loss of global coherence.

3. **Memory vs. Efficiency Tradeoff**  
   Larger context windows dramatically increase memory and compute usage—scaling roughly quadratically with context length in attention-based models (like Transformers). So there's a cost: doubling the context may more than double resource demands.

4. **Real-World Applicability**  
   In applications like legal/medical analysis, customer support, or research assistance, users expect models to handle full documents, multi-turn nuanced dialogues, or complex reasoning chains. Limited context restricts utility.

5. **Prompt Engineering Flexibility**  
   With more context, you can include rich instructions, examples (few-shot learning), and background knowledge *within a single prompt*, improving reliability and reducing need for external tools.

### Example:
- **GPT-3**: ~2K–4K tokens  
- **GPT-4 (standard)**: 8K tokens  
- **GPT-4 Turbo / Claude 3.5 Sonnet**: Up to 200K tokens  
→ A 200K-token context can hold ~150 pages of text, enabling analysis of an entire book or long conversation history.

In short: **larger context windows = more human-like understanding over extended text, but with significant computational cost**—making them a critical tradeoff for model design and deployment.

---
## Chapter 3 — Context is a Budget

**Goal:** understand *why* context management exists before we build any of it.

Every token you load — user message, system prompt, file content, conversation history — occupies space in the context window. When that space runs out, one of two things happens: the model silently truncates early content (it forgets the beginning of the conversation), or the API returns an error.

A coding agent that naively loads every file in a repo will hit the wall on the first non-trivial query.

**You will:**
- Replace the inline `len // 4` guess with a named `count_tokens()` function
- Watch a multi-turn conversation burn through its budget turn by turn
- Measure the token cost of real files on disk
- See the panel turn yellow then red as budget pressure rises
- Understand the three strategies agents use to stay inside the window (foreshadowing Ch 3–8)

### 3.1 `count_tokens()` — one place to estimate token cost

If we were using OpenAI models, we could use tiktoken for exact token counts.
For a provider‑agnostic approach, we’d need a tokenizer from the model’s ecosystem (often a Hugging Face tokenizer) — which is heavier to load and configure.

The `÷ 4` heuristic (1 token ≈ 4 characters in English prose) is accurate to
within ~15 % for most code and documentation. That's precise enough to drive
a budget bar — we don't need exact counts, we need *early warning*.

Wrapping it in a named function means we can swap the implementation once
(e.g. call Ollama's `/api/tokenize` endpoint) without touching any of the
chapter code that calls it.

In [21]:
def count_tokens(text: str) -> int:
    """
    Estimate token count for *text*.

    Uses the 4-characters-per-token heuristic — accurate to ~15 % for
    English prose and source code.  Good enough to drive a budget bar.

    Swap the body for a real tokeniser call if you need precision:
        # example: use Ollama's tokenise endpoint
        # r = requests.post(f"{OLLAMA_BASE_URL}/api/tokenize",
        #                   json={"model": OLLAMA_MODEL, "content": text})
        # return len(r.json()["tokens"])
    """
    return max(1, len(text) // 4)


# Quick sanity check
_sample = "The quick brown fox jumps over the lazy dog."
print(f"Sample: {len(_sample)} chars → {count_tokens(_sample)} tokens  "
      f"(GPT-4 tokeniser gives ~11 for this sentence)")

Sample: 44 chars → 11 tokens  (GPT-4 tokeniser gives ~11 for this sentence)


### 3.2 Watching a conversation burn its budget

Each turn appends the user message *and* the assistant reply to the
`messages` list before the next call.  Ollama sees the full list every time —
that's how it maintains conversation context, but it also means every reply
you get back is added to your next prompt.

Run this cell and watch the budget bar grow with each exchange.

In [22]:
TURNS = [
    "What is a Python list comprehension? Give a one-line example.",
    "Now show a dictionary comprehension that squares numbers 1–5.",
    "What is the difference between a shallow copy and a deep copy?",
    "Show me a minimal example where a shallow copy causes a bug.",
]

messages: list[dict] = []
tokens_used = 0

for i, question in enumerate(TURNS, start=1):
    messages.append({"role": "user", "content": question})

    # Estimate prompt size before the call
    prompt_text = " ".join(m["content"] for m in messages)
    prompt_size = count_tokens(prompt_text)

    show_rule(f"Turn {i} — before call")
    show_panel(
        query        = question,
        token_used   = tokens_used,
        strategy     = "multi-turn",
        prompt_size  = prompt_size,
    )

    reply, tokens_used = chat_continued(messages)
    messages.append({"role": "assistant", "content": reply})

    show_rule(f"Turn {i} — after call")
    show_panel(
        query       = question,
        token_used  = tokens_used,
        strategy    = "multi-turn",
        prompt_size = prompt_size,
    )
    print(f"\nReply: {reply[:200]}{'…' if len(reply) > 200 else ''}\n")


Token Budget,"0 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,15 tokens


Token Budget,"86 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,15 tokens



Reply: A Python list comprehension is a concise syntax for creating a new list by applying an expression to each item in an iterable, optionally filtering items with a condition.

**One-line example:**
```py…



Token Budget,"86 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,95 tokens


Token Budget,"133 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,95 tokens


...

Token Budget,"349 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,126 tokens



Reply: A **shallow copy** creates a new outer container (e.g., list, dict) but *reuses references* to the same nested objects—so changes to nested mutable objects affect both copies.  
A **deep copy** recurs…



Token Budget,"349 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,318 tokens


Token Budget,"586 / 262,144 tokens"
Retrieved,none
Strategy,multi-turn
Prompt size,318 tokens



Reply: Here's a minimal example where a shallow copy leads to an unexpected bug:

```python
# Original list containing a mutable nested list
data = [[1, 2], [3, 4]]

# Shallow copy: creates new outer list, b…



### 3.3 What files actually cost

A coding agent loads source files to answer questions about them.
Let's measure what that costs in tokens.

We'll scan the files in `REPO_ROOT`, count their token cost, and show
how quickly a naive "load everything" strategy would exhaust the budget.

In [24]:
import os
from pathlib import Path

# File extensions we'd want to load for a coding question
CODE_EXTENSIONS = {".py", ".js", ".ts", ".go", ".rs", ".java",
                   ".c", ".cpp", ".h", ".md", ".txt", ".yaml", ".toml", ".json"}

def scan_repo_costs(root: str) -> list[dict]:
    """
    Walk *root* and return a list of dicts:
        {"path": relative_path, "bytes": int, "tokens": int}
    for every source file found, sorted by token cost descending.
    """
    results = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames
                       if not d.startswith(".") and d not in
                       {"__pycache__", "node_modules", ".git", "venv", ".venv"}]
        for fname in filenames:
            full = Path(dirpath) / fname
            if full.suffix.lower() not in CODE_EXTENSIONS:
                continue
            try:
                text = full.read_text(errors="ignore")
            except OSError:
                continue
            results.append({
                "path":   str(full.relative_to(root)),
                "bytes":  len(text.encode()),
                "tokens": count_tokens(text),
            })
    return sorted(results, key=lambda x: x["tokens"], reverse=True)


file_costs = scan_repo_costs(REPO_ROOT)

# ── HTML table ──────────────────────────────────────────────────────────────
rows = ""
for f in file_costs[:20]:
    pct = f["tokens"] / TOKEN_BUDGET * 100
    color = "#2da44e" if pct < 10 else ("#bf8700" if pct < 30 else "#cf222e")
    rows += (
        f'<tr><td style="font-family:monospace;padding:3px 12px 3px 0">{f["path"]}</td>'
        f'<td style="text-align:right;padding:3px 8px">{f["bytes"]:,}</td>'
        f'<td style="text-align:right;padding:3px 8px;color:{color};font-weight:bold">{f["tokens"]:,}</td>'
        f'<td style="text-align:right;padding:3px 8px;color:{color}">{pct:.1f}%</td></tr>'
    )

total = sum(f["tokens"] for f in file_costs)
fits  = total <= TOKEN_BUDGET
summary_color = "#2da44e" if fits else "#cf222e"
summary = "YES ✓" if fits else "NO ✗"

display(HTML(f"""
<div style="font-family:monospace;font-size:0.88rem">
  <b>File token costs in '{REPO_ROOT}'</b>
  <table style="border-collapse:collapse;margin-top:8px">
    <tr style="border-bottom:1px solid #d0d7de;color:#57606a">
      <th style="text-align:left;padding:3px 12px 3px 0">File</th>
      <th style="text-align:right;padding:3px 8px">Bytes</th>
      <th style="text-align:right;padding:3px 8px">Tokens</th>
      <th style="text-align:right;padding:3px 8px">% of budget</th>
    </tr>
    {rows}
  </table>
  <div style="margin-top:8px;color:#57606a">
    Total across {len(file_costs)} files:&nbsp;
    <span style="color:{summary_color};font-weight:bold">{total:,} tokens</span>
    &nbsp;·&nbsp; Budget: {TOKEN_BUDGET:,}
    &nbsp;·&nbsp; Fits in one prompt:&nbsp;
    <span style="color:{summary_color};font-weight:bold">{summary}</span>
  </div>
</div>
"""))


File,Bytes,Tokens,% of budget
agent/core.py,"46,199","10,445",4.0%
agent copy/core.py,"46,199","10,445",4.0%
agent/app.py,"21,145","4,807",1.8%
agent copy/app.py,"19,458","4,374",1.7%
README.md,"6,941","1,723",0.7%
sample_project/tests/test_formatter_gen.py,"2,651",662,0.3%
AGENTS.md,"1,323",326,0.1%
sample_project/utils/parser.py,"1,221",303,0.1%
sample_project/tests/test_parser.py,"1,116",279,0.1%
sample_project/utils/validator.py,"1,044",261,0.1%


### 3.4 Hitting the wall — what over-budget looks like

Let's simulate what happens when a naive agent stuffs too much into one prompt.
We'll build a fake "load everything" prompt, measure it, and show the panel
turning red — without actually sending it to Ollama (no point wasting tokens
on a prompt that would be truncated or error).

In [26]:
def _simulate_overbudget() -> None:
    """
    Build a hypothetical prompt that exceeds TOKEN_BUDGET and show
    the panel at 50 %, 85 %, and 110 % capacity — no LLM call needed.
    """
    # Synthetic file list — imagine a mid-size Python project
    fake_files = [
        {"name": "src/parser.py",      "tokens": int(TOKEN_BUDGET * 1.18)},
        {"name": "src/compiler.py",    "tokens": int(TOKEN_BUDGET * 1.22)},
        {"name": "src/runtime.py",     "tokens": int(TOKEN_BUDGET * 1.20)},
        {"name": "tests/test_parser.py","tokens": int(TOKEN_BUDGET * 1.15)},
        {"name": "docs/architecture.md","tokens": int(TOKEN_BUDGET * 1.12)},
        {"name": "README.md",          "tokens": int(TOKEN_BUDGET * 1.08)},
        {"name": "setup.py",           "tokens": int(TOKEN_BUDGET * 1.05)},
    ]

    loaded_files = []
    accumulated  = 0
    query        = "Explain how the compiler hands off to the runtime"

    for i, f in enumerate(fake_files):
        accumulated += f["tokens"]
        loaded_files.append({"path": f["name"], "hot": True})

        pct = accumulated / TOKEN_BUDGET
        label = (
            "comfortable" if pct < 1.6 else
            "warning"     if pct < 1.85 else
            "OVER BUDGET"
        )
        show_rule(f"After loading {i+1} file(s) — {label}")
        show_panel(
            query        = query,
            token_used   = accumulated,
            files        = loaded_files,
            strategy     = "naive load-all",
            prompt_size  = accumulated,
        )

        if accumulated > TOKEN_BUDGET:
            print("\n⚠  Prompt exceeds context window.")
            print("Ollama will truncate the earliest messages — the agent may")
            print("forget the system prompt or earlier file contents entirely.\n")
            break

_simulate_overbudget()

Token Budget,"309,329 / 262,144 tokens"
Retrieved,HOT src/parser.py
Strategy,naive load-all
Prompt size,"309,329 tokens"



⚠  Prompt exceeds context window.
Ollama will truncate the earliest messages — the agent may
forget the system prompt or earlier file contents entirely.



### 3.5 The three strategies — what's coming next

Every technique in Chapters 3–8 is a variation of one of these three responses
to budget pressure:

| Strategy | What it does | Where we build it |
|----------|-------------|-------------------|
| **Be selective** | Only load files relevant to the query | Ch 3 (manifest), Ch 4 (glob), Ch 5 (fuzzy), Ch 6 (grep) |
| **Be lazy** | Load file *summaries* first, full content only on demand | Ch 4 JIT reads |
| **Evict & compress** | Move cold context to a summary store; keep hot context small | Ch 7 microcompaction, Ch 8 semantic RAG |

The key insight: **a good agent spends tokens like a good programmer spends CPU** — only when necessary, and on the thing most likely to matter.

Chapter 4 introduces the first tool: an `AGENTS.md` manifest that tells the agent which files matter *before* it even looks at the file tree.

---
## Chapter 4 — Give it a Map

**Goal:** before the agent looks at a single file, hand it a curated index of the repo so it already knows what exists and what matters.

That index is `AGENTS.md` — a small markdown file you commit at the repo root.
It costs a fixed, known number of tokens on every call (cheap), and it dramatically reduces the search space for every retrieval strategy we build later.

**You will:**
- Create an `AGENTS.md` for this repo
- Write `load_manifest()` — reads the file, extracts mentioned paths
- Write `ask_with_manifest()` — prepends the manifest to every prompt
- See the panel report the manifest's token cost before any file is loaded

### 4.1 What goes in AGENTS.md?

An `AGENTS.md` is just a markdown file. It has no required schema — the agent
reads it as plain text. The convention is:

- **Entry points** — where execution starts
- **Key modules** — what each important file does in one line
- **Ownership sections** — "questions about X → look in Y"
- **Off-limits** — generated files, build artefacts the agent should skip

Keep it under ~400 tokens. If it grows larger than that, it starts eating the
budget it was meant to protect.

The cell below uses `%%writefile` to create `AGENTS.md` on disk.
`%%writefile path` is a Jupyter magic: instead of running the cell,
it saves the cell body verbatim to the given path. The file will live next to
this notebook in `REPO_ROOT`.

In [27]:
%%writefile AGENTS.md
# AGENTS.md — Pocket Agent project map

## What this repo is
A Jupyter notebook tutorial that builds a local coding agent step by step.
Each chapter adds one capability. The notebook IS the source of truth.

## Entry point
- `pocket_agent.ipynb` — the entire project lives here

## Chapter guide
| Chapter | Topic | Key concepts defined |
|---------|-------|----------------------|
| 1 | Hello Ollama | `chat()`, `show_panel()`, `ping_ollama()` |
| 2 | Context budget | `count_tokens()`, `scan_repo_costs()` |
| 3 | Manifest | `load_manifest()`, `ask_with_manifest()` |
| 4 | Glob + JIT reads | `glob_files()`, `jit_read()` |
| 5 | Fuzzy scoring | `score_files()` |
| 6 | Grep | `grep_repo()` |
| 7 | Microcompaction | `compact()`, hot/cold store |
| 8 | Semantic RAG | `embed()`, `retrieve()` |
| 9 | Full pipeline | `run()` |
| 9b | Web UI | FastAPI + WebSocket server |
| 10 | Write + diff | `write_file()`, `make_diff()` |
| 11 | Agent loop | `agent_loop()` |
| 12 | Test generation | `generate_tests()` |

## Off-limits (never load these)
- `.git/` — git internals
- `__pycache__/` — compiled bytecode
- `*.ipynb_checkpoints/` — Jupyter autosave noise

## Questions about token budgeting → Chapter 3
## Questions about retrieval strategy → Chapters 4–8
## Questions about the agent loop → Chapter 12

Overwriting AGENTS.md


### 4.2 `load_manifest()` — read the map

`load_manifest()` does two things:
1. Reads `AGENTS.md` as plain text (to inject into the prompt)
2. Extracts any file paths mentioned in it (for later retrieval stages to use as hints)

The path extraction is intentionally simple — a regex that finds things
that look like `path/to/file.ext`. False positives are fine; this is
advisory, not authoritative.

In [28]:
import re

MANIFEST_FILENAME = "AGENTS.md"

def load_manifest(repo_root: str = REPO_ROOT) -> dict:
    """
    Read AGENTS.md from *repo_root*.

    Returns a dict:
        {
          "text":   str,        # full file content, ready to inject into a prompt
          "tokens": int,        # estimated token cost
          "paths":  list[str],  # file paths mentioned in the manifest
          "found":  bool,       # False if the file doesn't exist
        }
    """
    manifest_path = Path(repo_root) / MANIFEST_FILENAME

    if not manifest_path.exists():
        return {"text": "", "tokens": 0, "paths": [], "found": False}

    text = manifest_path.read_text(errors="ignore")

    # Extract things that look like file paths: word chars, slashes, dots
    # e.g.  pocket_agent.ipynb  src/parser.py  docs/architecture.md
    paths = re.findall(r'\b[\w./\-]+\.[\w]+\b', text)
    # Deduplicate while preserving order
    seen, unique_paths = set(), []
    for p in paths:
        if p not in seen:
            seen.add(p)
            unique_paths.append(p)

    return {
        "text":   text,
        "tokens": count_tokens(text),
        "paths":  unique_paths,
        "found":  True,
    }


manifest = load_manifest()
print(f"Manifest found : {manifest['found']}")
print(f"Token cost     : {manifest['tokens']}  ({manifest['tokens']/TOKEN_BUDGET*100:.1f}% of budget)")
print(f"Paths mentioned: {manifest['paths'][:8]}")

Manifest found : True
Token cost     : 326  (0.1% of budget)
Paths mentioned: ['AGENTS.md', 'pocket_agent.ipynb']


### 4.3 `ask_with_manifest()` — the manifest-aware query

Every prompt the agent sends from now on follows this structure:

```
[SYSTEM]
You are a coding assistant. Here is the project map:
<AGENTS.md contents>

[USER]
<query>
```

The manifest is injected once, costs a fixed number of tokens, and gives
the model the project layout before it has to reason about anything.

In [29]:
def ask_with_manifest(
    query:     str,
    repo_root: str = REPO_ROOT,
    files:     list[dict] | None = None,
) -> tuple[str, int]:
    """
    Send *query* to Ollama with the project manifest prepended as a system prompt.

    Parameters
    ----------
    query     : the user's question
    repo_root : where to find AGENTS.md
    files     : already-loaded file dicts {"path", "content", "hot"}
                their content is appended after the manifest

    Returns (reply_text, tokens_used).
    """
    manifest = load_manifest(repo_root)
    files    = files or []

    # Build system prompt
    if manifest["found"]:
        system_content = (
            "You are a coding assistant with access to the project map below.\n"
            "Use it to understand the codebase structure before answering.\n\n"
            f"--- PROJECT MAP (AGENTS.md) ---\n{manifest['text']}\n---"
        )
    else:
        system_content = "You are a coding assistant."

    # Append any loaded file contents
    file_block = ""
    for f in files:
        file_block += f"\n\n--- FILE: {f['path']} ---\n{f.get('content', '')}"

    messages = [
        {"role": "system",  "content": system_content},
        {"role": "user",    "content": query + file_block},
    ]

    prompt_size = count_tokens(system_content + query + file_block)

    show_panel(
        query        = query,
        token_used   = prompt_size,
        files        = [{"path": f["path"], "hot": f.get("hot", True)} for f in files],
        strategy     = "manifest",
        prompt_size  = prompt_size,
    )

    reply, tokens_used = chat_continued(messages)
    return reply, tokens_used

### 4.4 Try it

Ask a question about the project. The manifest is in the prompt so the model
already knows the chapter structure before it replies.

In [30]:
query = "Which chapter should I look at to understand how retrieval works?"

show_rule("Chapter 4 — manifest-guided query")
reply, tokens_used = ask_with_manifest(query)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    strategy    = "manifest",
    prompt_size = count_tokens(query),
)
show_reply(reply)


Token Budget,"383 / 262,144 tokens"
Retrieved,none
Strategy,manifest
Prompt size,383 tokens


Token Budget,"549 / 262,144 tokens"
Retrieved,none
Strategy,manifest
Prompt size,16 tokens


To understand how retrieval works, look at **Chapter 8: Semantic RAG**, which covers:

- `embed()` — for generating embeddings  
- `retrieve()` — for semantic search and retrieval

These functions implement retrieval-augmented generation (RAG), which uses vector similarity to fetch relevant context from a repository.

You may also want to briefly review **Chapters 4–7** first, since they lay the groundwork (globs, JIT reads, fuzzy scoring, and grep), which inform how retrieval fits into the full pipeline in **Chapter 9**.

---
## Chapter 5 — Glob + JIT Reads

**Goal:** navigate a file tree without bulk-loading it.

Chapter 4 gave the agent a curated map. But what about files the map doesn't
mention, or repos where no `AGENTS.md` exists? The agent needs to *find* files
on its own — without reading them all.

The answer is a two-phase approach:

1. **Glob** — list every file that *could* be relevant (filenames only, no content).  
   This is essentially free: no tokens spent yet.
2. **JIT read** — load each file's content *just in time*, one at a time,  
   and stop when the token budget would tip past a safe threshold.

Files that were found but not loaded appear as **COLD** in the panel.
Files whose content is in the prompt are **HOT**.

**You will:**
- Create a small sample project to give glob something to work with
- Write `glob_files()` — match file paths against a pattern
- Write `jit_read()` — load one file on demand
- Write `budget_load()` — greedily load HOT files until budget threshold
- See the panel split between HOT and COLD for the first time

### 5.1 Sample project

Our repo only has one notebook and `AGENTS.md` — not enough to demonstrate
file navigation. The cells below use `%%writefile` to create a small fake
Python project under `sample_project/`.

Run each cell once; the files persist on disk for all later chapters.

In [ ]:
from typing import NamedTuple


class RunResult(NamedTuple):
    reply:       str
    strategy:    str
    files:       list[dict]   # full list, HOT and COLD
    tokens_used: int
    compact_log: list[str]    # empty if compaction didn't fire


def run(
    query:     str,
    repo_root: str  = REPO_ROOT,
    strategy:  str  = "auto",   # "auto" | "fuzzy" | "grep" | "semantic"
    top_k:     int  = 8,
) -> RunResult:
    """
    Full retrieval + generation pipeline.

    Parameters
    ----------
    query     : the user's question
    repo_root : path to the repository to search
    strategy  : retrieval strategy; "auto" lets pick_strategy() decide
    top_k     : max candidates to consider before budget_load()
    """
    # ── 1. Manifest ─────────────────────────────────────────────────────────
    manifest     = load_manifest(repo_root)    # from the target repo, not REPO_ROOT
    manifest_tok = manifest["tokens"]

    # ── 2. Strategy selection ────────────────────────────────────────────────
    strat = pick_strategy(query) if strategy == "auto" else strategy

    # ── 3. Retrieval ─────────────────────────────────────────────────────────
    if strat == "grep":
        candidates = grep_rank(query, repo_root=repo_root)
        # Append fuzzy extras for files with zero grep hits
        found_paths = {f["path"] for f in candidates}
        extras = [f for f in rank_files(glob_files("*.py", repo_root=repo_root), query)
                  if f["path"] not in found_paths]
        candidates = (candidates + extras)[:top_k]

    elif strat == "semantic":
        candidates = semantic_retrieve(query, repo_root=repo_root, top_k=top_k)

    else:   # fuzzy
        candidates = rank_files(glob_files("*.py", repo_root=repo_root), query)[:top_k]

    # ── 4. Budget-aware JIT load ─────────────────────────────────────────────
    loaded = budget_load(candidates, already_used=manifest_tok, repo_root=repo_root)

    # ── 5. Compaction (if needed) ─────────────────────────────────────────────
    hot_tok = sum(f.get("tokens", 0) for f in loaded if f["hot"])
    total   = manifest_tok + hot_tok + count_tokens(query)
    loaded, total, compact_log = compact(loaded, query, total)

    # ── 6. Show panel ─────────────────────────────────────────────────────────
    hot_files = [f for f in loaded if f["hot"]]
    show_panel(
        query        = query,
        token_used   = total,
        files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
        strategy     = strat,
        prompt_size  = total,
    )

    # ── 7. Generate reply ─────────────────────────────────────────────────────
    reply, tokens_used = ask_with_manifest(query, repo_root=repo_root, files=hot_files)

    return RunResult(
        reply        = reply,
        strategy     = strat,
        files        = loaded,
        tokens_used  = tokens_used,
        compact_log  = compact_log,
    )

In [ ]:
%%writefile sample_project/main.py
"""Entry point for the JSON-to-CSV converter tool."""

from utils.parser import parse_json
from utils.formatter import to_csv
from utils.validator import validate_schema
import sys


def main(input_path: str, output_path: str, schema_path: str | None = None) -> None:
    with open(input_path) as f:
        raw = f.read()

    records = parse_json(raw)

    if schema_path:
        with open(schema_path) as f:
            schema = f.read()
        validate_schema(records, schema)

    csv_text = to_csv(records)

    with open(output_path, "w") as f:
        f.write(csv_text)

    print(f"Wrote {len(records)} records to {output_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])

In [ ]:
%%writefile sample_project/utils/parser.py
"""Parse a JSON string into a list of flat dicts."""

import json


def parse_json(raw: str) -> list[dict]:
    """
    Accept a JSON string that is either:
    - a list of objects   → returned as-is
    - a single object     → wrapped in a list
    - a list of scalars   → each scalar wrapped as {"value": scalar}

    Raises ValueError on malformed input.
    """
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON: {exc}") from exc

    if isinstance(data, list):
        return [_flatten(item) if isinstance(item, dict) else {"value": item}
                for item in data]
    if isinstance(data, dict):
        return [_flatten(data)]

    raise ValueError(f"Expected a JSON object or array, got {type(data).__name__}")


def _flatten(obj: dict, prefix: str = "", sep: str = ".") -> dict:
    """Recursively flatten nested dicts: {"a": {"b": 1}} → {"a.b": 1}."""
    result = {}
    for key, value in obj.items():
        full_key = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            result.update(_flatten(value, full_key, sep))
        else:
            result[full_key] = value
    return result

In [ ]:
%%writefile sample_project/utils/formatter.py
"""Format a list of flat dicts as CSV text."""

import csv
import io


def to_csv(records: list[dict], delimiter: str = ",") -> str:
    """
    Convert *records* (list of flat dicts) to a CSV string.

    All keys across all records are used as headers.
    Missing values are rendered as empty strings.
    """
    if not records:
        return ""

    # Collect all headers preserving first-seen order
    headers: list[str] = []
    seen: set[str] = set()
    for rec in records:
        for key in rec:
            if key not in seen:
                headers.append(key)
                seen.add(key)

    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=headers,
                            delimiter=delimiter, extrasaction="ignore")
    writer.writeheader()
    for rec in records:
        writer.writerow({h: rec.get(h, "") for h in headers})

    return buf.getvalue()

In [ ]:
%%writefile sample_project/utils/validator.py
"""Validate a list of records against a simple JSON schema."""

import json


def validate_schema(records: list[dict], schema_raw: str) -> None:
    """
    Validate each record against *schema_raw* (a JSON object mapping
    field names to expected types: "string", "number", "boolean").

    Raises TypeError on the first violation found.
    """
    schema: dict[str, str] = json.loads(schema_raw)
    type_map = {"string": str, "number": (int, float), "boolean": bool}

    for i, record in enumerate(records):
        for field, expected_type_name in schema.items():
            if field not in record:
                raise TypeError(
                    f"Record {i}: missing required field '{field}'"
                )
            expected = type_map.get(expected_type_name)
            if expected and not isinstance(record[field], expected):
                raise TypeError(
                    f"Record {i}: field '{field}' expected {expected_type_name}, "
                    f"got {type(record[field]).__name__}"
                )

In [ ]:
%%writefile sample_project/tests/test_parser.py
"""Unit tests for utils/parser.py."""

import pytest
from utils.parser import parse_json, _flatten


class TestParseJson:
    def test_list_of_dicts(self):
        result = parse_json('[{"a": 1}, {"a": 2}]')
        assert result == [{"a": 1}, {"a": 2}]

    def test_single_dict_wrapped(self):
        result = parse_json('{"x": 10}')
        assert result == [{"x": 10}]

    def test_list_of_scalars(self):
        result = parse_json('[1, 2, 3]')
        assert result == [{"value": 1}, {"value": 2}, {"value": 3}]

    def test_invalid_json_raises(self):
        with pytest.raises(ValueError, match="Invalid JSON"):
            parse_json("{not valid}")

    def test_unexpected_type_raises(self):
        with pytest.raises(ValueError, match="Expected"):
            parse_json('"just a string"')


class TestFlatten:
    def test_nested_dict(self):
        assert _flatten({"a": {"b": 1}}) == {"a.b": 1}

    def test_deeply_nested(self):
        assert _flatten({"a": {"b": {"c": 42}}}) == {"a.b.c": 42}

    def test_flat_dict_unchanged(self):
        assert _flatten({"x": 1, "y": 2}) == {"x": 1, "y": 2}

In [ ]:
%%writefile sample_project/tests/test_formatter.py
"""Unit tests for utils/formatter.py."""

from utils.formatter import to_csv


def test_empty_input():
    assert to_csv([]) == ""


def test_single_record():
    result = to_csv([{"name": "Alice", "age": 30}])
    lines = result.strip().splitlines()
    assert lines[0] == "name,age"
    assert lines[1] == "Alice,30"


def test_missing_fields_become_empty():
    records = [{"a": 1, "b": 2}, {"a": 3}]
    result = to_csv(records)
    lines = result.strip().splitlines()
    assert lines[0] == "a,b"
    assert lines[2] == "3,"


def test_header_order_follows_first_seen():
    records = [{"z": 1, "a": 2}]
    result = to_csv(records)
    assert result.startswith("z,a")

### 5.2 `glob_files()` — list without loading

`glob_files()` returns a list of matching file paths and their sizes —
but **no file content**. Zero tokens spent so far.

The `pattern` argument follows Python's `fnmatch` syntax:  
`*.py` matches any `.py` file, `utils/*.py` matches only files directly in `utils/`.

In [ ]:
import fnmatch

SKIP_DIRS = {".git", "__pycache__", "node_modules", ".venv", "venv",
             ".ipynb_checkpoints", ".mypy_cache", ".pytest_cache"}


def glob_files(
    pattern:   str,
    repo_root: str = REPO_ROOT,
) -> list[dict]:
    """
    Walk *repo_root* and return every file whose name matches *pattern*.

    Returns a list of dicts:
        {"path": str,   # relative to repo_root
         "bytes": int,  # file size — no content loaded
         "hot":   bool} # always False at this stage
    Sorted by path for deterministic ordering.
    """
    matches = []
    root = Path(repo_root)

    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        for fname in filenames:
            if fnmatch.fnmatch(fname, pattern):
                full = Path(dirpath) / fname
                try:
                    size = full.stat().st_size
                except OSError:
                    continue
                matches.append({
                    "path":  str(full.relative_to(root)),
                    "bytes": size,
                    "hot":   False,
                })

    return sorted(matches, key=lambda x: x["path"])


# ── Demo ────────────────────────────────────────────────────────────────────
found = glob_files("*.py", repo_root="sample_project")
print(f"Found {len(found)} Python files:\n")
for f in found:
    print(f"  {f['path']:45s}  {f['bytes']:>5} bytes")

### 5.3 `jit_read()` — load one file on demand

`jit_read()` takes a path from the glob list and reads its content.
It returns the same dict shape, extended with `"content"` and `"tokens"`,
and flips `"hot"` to `True`.

In [ ]:
def jit_read(file_meta: dict, repo_root: str = REPO_ROOT) -> dict:
    """
    Read the content of one file described by *file_meta* (a glob result dict).

    Returns a new dict with the same keys plus:
        "content": str   — full file text
        "tokens":  int   — estimated token cost
        "hot":     True  — this file is now in the prompt
    """
    full_path = Path(repo_root) / file_meta["path"]
    try:
        content = full_path.read_text(errors="ignore")
    except OSError as exc:
        content = f"[error reading file: {exc}]"

    return {
        **file_meta,
        "content": content,
        "tokens":  count_tokens(content),
        "hot":     True,
    }


# ── Demo: read one file ─────────────────────────────────────────────────────
sample_meta = glob_files("parser.py", repo_root="sample_project")[0]
loaded      = jit_read(sample_meta, repo_root="sample_project")

print(f"Path   : {loaded['path']}")
print(f"Tokens : {loaded['tokens']}")
print(f"Hot    : {loaded['hot']}")
print(f"\nFirst 3 lines:\n{''.join(loaded['content'].splitlines(keepends=True)[:3])}")

### 5.4 `budget_load()` — greedy HOT/COLD split

Now we combine glob and JIT read.
`budget_load()` takes a list of glob results, reads them one by one,
and stops before the token budget hits a safety threshold.
Files it couldn't fit are kept in the list as COLD.

The threshold is 1.7 (70 % of budget) — leaving room for the manifest,
the query, and the model's reply.

In [ ]:
HOT_THRESHOLD = 1.70   # stop loading when prompt would exceed 70 % of budget


def budget_load(
    candidates: list[dict],
    already_used: int      = 0,
    repo_root:    str      = REPO_ROOT,
    threshold:    float    = HOT_THRESHOLD,
) -> list[dict]:
    """
    JIT-read files from *candidates* until adding the next one would push
    the running token total past *threshold* × TOKEN_BUDGET.

    Returns the full candidate list with:
      - loaded files marked  hot=True  and populated with "content"/"tokens"
      - unloaded files kept  hot=False (no content key added)
    """
    budget_limit = int(TOKEN_BUDGET * threshold)
    used         = already_used
    result       = []

    for meta in candidates:
        headroom = budget_limit - used
        # Peek at size without reading: use byte count as a cheap proxy
        estimated = meta["bytes"] // 4
        if estimated > headroom:
            result.append({**meta, "hot": False})
            continue

        loaded = jit_read(meta, repo_root=repo_root)
        if used + loaded["tokens"] > budget_limit:
            result.append({**meta, "hot": False})
        else:
            used += loaded["tokens"]
            result.append(loaded)

    return result


# ── Demo: load sample_project with a tiny artificial budget ─────────────────
candidates = glob_files("*.py", repo_root="sample_project")
loaded_files = budget_load(candidates, repo_root="sample_project")

hot  = [f for f in loaded_files if f["hot"]]
cold = [f for f in loaded_files if not f["hot"]]

show_rule("budget_load() result")
show_panel(
    query        = "How does the project handle nested JSON objects?",
    token_used   = sum(f.get("tokens", 0) for f in hot),
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + JIT",
    prompt_size  = sum(f.get("tokens", 0) for f in hot),
)
print(f"\nHOT ({len(hot)} files, {sum(f['tokens'] for f in hot):,} tokens):")
for f in hot:
    print(f"  ✓ {f['path']}")
print(f"\nCOLD ({len(cold)} files — found but not loaded):")
for f in cold:
    print(f"  ✗ {f['path']}")

### 5.5 Try it end-to-end

Ask a question about the sample project.
The agent globs for Python files, JIT-loads what fits in the budget,
and sends the HOT files plus the manifest to Ollama.

In [ ]:
query      = "How does the parser handle a JSON array of plain numbers like [1, 2, 3]?"
repo       = "sample_project"

# Phase 1: glob — free, no tokens spent
candidates   = glob_files("*.py", repo_root=repo)

# Phase 2: budget-aware JIT load
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files    = [f for f in loaded_files if f["hot"]]
prompt_size  = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 5 — before LLM call")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + JIT",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "glob + JIT",
    prompt_size = prompt_size,
)
show_reply(reply)


---
## Chapter 6 — Fuzzy Scoring

**Goal:** rank retrieved files before loading them, so budget_load() always drops the *least* relevant files when it runs out of room.

Chapter 5's glob returns files in alphabetical order.
If the budget fills up, it drops whatever comes last alphabetically — which may be the most relevant file.
Fuzzy scoring fixes this by sorting candidates by relevance *before* the JIT load loop.

Scoring works entirely on **file paths** — no content is read.
That keeps it free (no tokens, no disk I/O beyond what glob already did).

**Signals used:**
| Signal | Weight |
|--------|--------|
| Exact query word in filename stem | high |
| Fuzzy match between query word and filename stem | medium |
| Query word appears in a directory component | low |
| File is a test file, query doesn't mention tests | penalty ×1.5 |

**You will:**
- Write `tokenize_query()` — normalise a query into scoreable terms
- Write `score_file()` — score one file against the term list
- Write `rank_files()` — sort a candidate list by score descending
- Replace the alphabetical glob order with ranked order and see the panel change

### 6.1 `tokenize_query()` — extract scoreable terms

We strip stop words and short tokens so scores aren't diluted by
words like "the", "a", "how", "does".

In [ ]:
import difflib

_STOP_WORDS = {
    "a", "an", "the", "is", "in", "it", "of", "to", "do", "does",
    "how", "what", "why", "when", "where", "which", "who", "for",
    "and", "or", "but", "not", "with", "this", "that", "are", "was",
    "i", "me", "my", "we", "our", "you", "your",
}


def tokenize_query(query: str) -> list[str]:
    """
    Lowercase, split on non-alphanumeric chars, remove stop words and
    tokens shorter than 3 characters.

    "How does the formatter handle missing fields?"
    → ["formatter", "handle", "missing", "fields"]
    """
    words = re.split(r"[^a-zA-Z0-9]+", query.lower())
    return [w for w in words if len(w) >= 3 and w not in _STOP_WORDS]


# Quick check
print(tokenize_query("How does the formatter handle missing fields?"))
print(tokenize_query("Where is the CSV delimiter configured?"))

### 6.2 `score_file()` — score one file against a query

`difflib.SequenceMatcher` gives us a similarity ratio between 0 and 1
with no extra dependencies — it's in the Python standard library.

In [ ]:
def score_file(meta: dict, query_terms: list[str]) -> float:
    """
    Score *meta* (a glob result dict) against *query_terms*.

    Higher = more relevant.  Returns 1.0 for an empty term list.
    No file content is read — scoring is path-only.
    """
    if not query_terms:
        return 1.0

    path  = meta["path"].lower()
    parts = Path(path).parts          # ("utils", "parser.py")
    stem  = Path(path).stem           # "parser"
    dirs  = parts[:-1]                # ("utils",)

    score = 1.0
    for term in query_terms:
        # ── Filename stem ────────────────────────────────────────────
        if term == stem:
            score += 4.0                          # exact match
        elif term in stem or stem in term:
            score += 2.5                          # substring match
        else:
            ratio = difflib.SequenceMatcher(None, term, stem).ratio()
            score += ratio * 2.0                  # fuzzy match

        # ── Directory components ──────────────────────────────────────
        for d in dirs:
            if term == d:
                score += 2.0
            elif term in d:
                score += 1.5
            else:
                ratio = difflib.SequenceMatcher(None, term, d).ratio()
                score += ratio * 1.3

    # ── Test-file penalty ─────────────────────────────────────────────
    if "test" in path and "test" not in query_terms:
        score *= 1.5

    return round(score, 4)


# ── Sanity check ─────────────────────────────────────────────────────────────
_candidates = glob_files("*.py", repo_root="sample_project")
_terms      = tokenize_query("How does the formatter handle missing fields?")

for f in _candidates:
    print(f"{score_file(f, _terms):.4f}  {f['path']}")

### 6.3 `rank_files()` — sort the candidate list

A thin wrapper that applies `score_file()` to every candidate and
returns them sorted highest-score-first.
Files with a score of 0 are kept — they're still candidates,
just lowest priority.

In [ ]:
def rank_files(candidates: list[dict], query: str) -> list[dict]:
    """
    Return *candidates* sorted by relevance to *query*, highest first.
    Adds a "score" key to each dict.
    """
    terms = tokenize_query(query)
    scored = [
        {**c, "score": score_file(c, terms)}
        for c in candidates
    ]
    return sorted(scored, key=lambda x: x["score"], reverse=True)


# ── Show ranked order vs alphabetical ────────────────────────────────────────
query      = "How does the formatter handle missing fields?"
candidates = glob_files("*.py", repo_root="sample_project")
ranked     = rank_files(candidates, query)

rows = ""
for i, f in enumerate(ranked, 1):
    color = "#2da44e" if i == 1 else ("#bf8700" if i <= 3 else "#8c959f")
    rows += (
        f'<tr><td style="text-align:right;padding:3px 10px;color:#8c959f">{i}</td>\'
        f'<td style="text-align:right;padding:3px 10px;color:{color};font-weight:bold">{f["score"]:.4f}</td>\'
        f'<td style="padding:3px 10px;font-family:monospace">{f["path"]}</td></tr>'
    )

display(HTML(f"""
<div style="font-size:0.88rem">
  <b>Ranked candidates</b>
  <table style="border-collapse:collapse;margin-top:6px">
    <tr style="border-bottom:1px solid #d0d7de;color:#57606a">
      <th style="text-align:right;padding:3px 10px">Rank</th>
      <th style="text-align:right;padding:3px 10px">Score</th>
      <th style="text-align:left;padding:3px 10px">File</th>
    </tr>
    {rows}
  </table>
</div>
"""))


### 6.4 Try it — ranked glob + JIT

Same query as Chapter 5, but now candidates are ranked before budget_load()
sees them. The first file that turns COLD will be the least relevant, not a
random alphabetical casualty.

In [ ]:
query = "How does the formatter handle missing fields?"
repo  = "sample_project"

# Phase 1: glob
candidates = glob_files("*.py", repo_root=repo)

# Phase 2: rank  ← new
ranked = rank_files(candidates, query)

# Phase 3: budget-aware JIT load (now works on ranked order)
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(ranked, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 6 — fuzzy-ranked before LLM call")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "glob + fuzzy + JIT",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "glob + fuzzy + JIT",
    prompt_size = prompt_size,
)
show_reply(reply)


---
## Chapter 7 — Grep

**Goal:** find files by what's *inside* them, not just their name.

Chapters 4–5 rank files by path relevance. That works well when the
query mentions a filename — "how does the *formatter* work?" — but fails
when the relevant file has a generic name.

Consider: *"Where does the code raise a TypeError?"*
No filename contains "typeerror". Fuzzy scoring gives every file a near-zero
path score. But `validator.py` *contains* `raise TypeError(...)` — and grep
finds it instantly.

Grep is the third retrieval strategy. It searches file content for
patterns derived from the query, counts matches per file, and uses that
count as an additional relevance signal on top of the fuzzy path score.

**You will:**
- Write `grep_repo()` — search file contents, return matches with excerpts
- Write `query_to_patterns()` — turn query terms into search patterns
- Write `grep_rank()` — combine grep hit count with fuzzy path score
- See grep surface `validator.py` for a query that fuzzy scoring misses entirely

### 7.1 `grep_repo()` — search file contents

For each file, we search for a compiled regex pattern and collect:
- the number of matching lines (used for ranking)
- up to `context_lines` surrounding each match (used in the prompt as an excerpt)

Returning excerpts rather than full file content is a key budget trick:
if grep finds 2 matching lines in a 300-line file, we can show just those
2 lines plus context instead of loading all 300 lines HOT.

In [ ]:
def grep_repo(
    pattern:       str,
    repo_root:     str   = REPO_ROOT,
    extensions:    set   = CODE_EXTENSIONS,
    context_lines: int   = 2,
    max_matches:   int   = 5,
) -> list[dict]:
    """
    Search every matching file under *repo_root* for *pattern* (regex).

    Returns a list of dicts — one per file that had at least one match:
        {
          "path":      str,         # relative to repo_root
          "hit_count": int,         # number of matching lines
          "excerpt":   str,         # up to max_matches hits with context
          "bytes":     int,
          "hot":       False,       # budget_load() will flip this
          "score":     1.0,         # grep_rank() will fill this
        }
    Sorted by hit_count descending.
    """
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error:
        regex = re.compile(re.escape(pattern), re.IGNORECASE)

    results = []
    root    = Path(repo_root)

    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        for fname in filenames:
            full = Path(dirpath) / fname
            if full.suffix.lower() not in extensions:
                continue
            try:
                lines = full.read_text(errors="ignore").splitlines()
            except OSError:
                continue

            # Find matching line indices
            hit_indices = [i for i, ln in enumerate(lines) if regex.search(ln)]
            if not hit_indices:
                continue

            # Build excerpt: up to max_matches hits with context
            excerpts, seen_lines = [], set()
            for hit_i in hit_indices[:max_matches]:
                start = max(0, hit_i - context_lines)
                end   = min(len(lines), hit_i + context_lines + 1)
                block = []
                for ln_i in range(start, end):
                    if ln_i not in seen_lines:
                        prefix = "→ " if ln_i == hit_i else "  "
                        block.append(f"{prefix}{ln_i+1:4d} │ {lines[ln_i]}")
                        seen_lines.add(ln_i)
                excerpts.append("\n".join(block))

            excerpt_text = "\n\n".join(excerpts)

            results.append({
                "path":      str(full.relative_to(root)),
                "hit_count": len(hit_indices),
                "excerpt":   excerpt_text,
                "bytes":     full.stat().st_size,
                "hot":       False,
                "score":     1.0,
            })

    return sorted(results, key=lambda x: x["hit_count"], reverse=True)


# ── Demo: search for TypeError ───────────────────────────────────────────────
hits = grep_repo("TypeError", repo_root="sample_project")
for h in hits:
    print(f"{h['hit_count']} hit(s)  {h['path']}")
    print(h["excerpt"])
    print()

### 7.2 `query_to_patterns()` — derive search patterns from a query

We turn the query terms into a single alternation regex:
`formatter|missing|fields`

This is a liberal match — any file mentioning *any* term is a candidate.
We'll use hit count to separate strong matches from weak ones.

In [ ]:
def query_to_patterns(query: str) -> str:
    """
    Convert a natural-language query into a single regex pattern
    suitable for grep_repo().

    "Where does the code raise a TypeError?"
    → "typeerror|raise|code"   (terms ≥ 4 chars, lowercased, joined with |)

    Returns an empty string if no usable terms are found.
    """
    terms = [t for t in tokenize_query(query) if len(t) >= 4]
    if not terms:
        return ""
    return "|".join(re.escape(t) for t in terms)


# Check a few examples
for q in [
    "Where does the code raise a TypeError?",
    "How does the formatter handle missing fields?",
    "What is the CSV delimiter default value?",
]:
    print(f"  {q!r}\n  → {query_to_patterns(q)!r}\n")

### 7.3 `grep_rank()` — combine grep hits with fuzzy path score

A file can score high two ways:
- Many grep hits → strong content signal
- High fuzzy path score → strong name signal

`grep_rank()` normalises both signals to [0, 1] and adds them,
so a file with a relevant name *and* relevant content beats one with only one signal.

In [ ]:
def grep_rank(
    query:     str,
    repo_root: str = REPO_ROOT,
) -> list[dict]:
    """
    Full grep-based retrieval pipeline for *query*:
    1. Derive a regex pattern from the query terms
    2. Grep every source file for that pattern
    3. Score each hit file with both grep hit_count and fuzzy path score
    4. Normalise both signals to [0, 1] and sum them
    5. Return sorted by combined score descending
    Files with zero grep hits are excluded entirely.
    """
    pattern = query_to_patterns(query)
    if not pattern:
        return []
    hits = grep_repo(pattern, repo_root=repo_root)
    if not hits:
        return []
    terms          = tokenize_query(query)
    max_hits       = max(h["hit_count"] for h in hits) or 1
    max_path_score = max(score_file(h, terms) for h in hits) or 1
    ranked = []
    for h in hits:
        norm_grep = h["hit_count"]    / max_hits
        norm_path = score_file(h, terms) / max_path_score
        combined  = round(norm_grep + norm_path, 4)
        ranked.append({**h, "score": combined})
    return sorted(ranked, key=lambda x: x["score"], reverse=True)


# ── Demo ──────────────────────────────────────────────────────────────────────
query  = "Where does the code raise a TypeError?"
ranked = grep_rank(query, repo_root="sample_project")

rows = ""
for f in ranked:
    excerpt = (f["excerpt"].splitlines()[0] if f["excerpt"] else "").replace("<","&lt;")
    rows += (
        f'<tr>\'
        f'<td style="text-align:right;padding:3px 10px;color:#2da44e;font-weight:bold">{f["score"]}</td>\'
        f'<td style="text-align:right;padding:3px 10px">{f["hit_count"]}</td>\'
        f'<td style="padding:3px 10px;font-family:monospace">{f["path"]}</td>\'
        f'<td style="padding:3px 10px;font-family:monospace;color:#57606a;font-size:0.82rem">{excerpt}</td>\'
        f'</tr>'
    )

display(HTML(f"""
<div style="font-size:0.88rem">
  <b>grep_rank(): {query!r}</b>
  <table style="border-collapse:collapse;margin-top:6px">
    <tr style="border-bottom:1px solid #d0d7de;color:#57606a">
      <th style="text-align:right;padding:3px 10px">Score</th>
      <th style="text-align:right;padding:3px 10px">Hits</th>
      <th style="text-align:left;padding:3px 10px">File</th>
      <th style="text-align:left;padding:3px 10px">First match</th>
    </tr>
    {rows}
  </table>
</div>
"""))

fuzzy_only = rank_files(glob_files("*.py", repo_root="sample_project"), query)
print("\nFuzzy path scores for same query (for comparison):")
for f in fuzzy_only:
    print(f"  {f['score']:.4f}  {f['path']}")


### 7.4 Try it — grep-guided query with excerpt injection

One more trick: instead of loading the *full* file HOT, we can inject
just the grep **excerpt** for files that matched well but are expensive
to load in full. This stretches the budget further.

For this demo we load full content (the sample files are small), but
the `excerpt` field is there and Chapter 8 will start using it during compaction.

In [ ]:
query = "Where does the code raise a TypeError?"
repo  = "sample_project"

# Grep-ranked candidates (includes excerpt, excludes files with zero hits)
candidates = grep_rank(query, repo_root=repo)

# Fall back to fuzzy glob for files with zero grep hits
all_paths    = {f["path"] for f in candidates}
glob_extras  = [f for f in rank_files(glob_files("*.py", repo_root=repo), query)
                if f["path"] not in all_paths]
candidates   = candidates + glob_extras   # grep hits first, fuzzy extras after

# Budget-aware JIT load
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 7 — grep-ranked before LLM call")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "grep + fuzzy + JIT",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "grep + fuzzy + JIT",
    prompt_size = prompt_size,
)
show_reply(reply)


---
## Chapter 8 — Microcompaction

**Goal:** survive long sessions without hitting the context ceiling.

After several turns the conversation grows: loaded file contents, prior replies,
system prompts. Eventually the budget fills up and the model starts forgetting
the beginning of the conversation.

Microcompaction solves this with a simple rule:

> When the running token count exceeds a **compaction threshold**,
> take the coldest HOT files, summarise each one in a single LLM call,
> replace the full content with the summary, and mark them COLD.

The summary is much smaller than the original — typically 10–15 % of the
original token count. Budget is freed; the session continues.

```
Before compaction            After compaction
─────────────────            ────────────────
HOT  utils/parser.py  420t   COLD utils/parser.py  [summary: 45t]
HOT  utils/formatter  380t   HOT  utils/formatter  380t   ← kept
HOT  utils/validator  290t   HOT  utils/validator  290t   ← kept
─────────────────            ────────────────
total: 1090t                 total: 715t   (saved 375t)
```

**You will:**
- Write `eviction_candidates()` — pick which HOT files to evict
- Write `summarise_file()` — compress one file to a short summary via Ollama
- Write `compact()` — orchestrate eviction + summarisation, return updated file list
- Simulate a session running over budget and watch compaction kick in

### 8.1 `eviction_candidates()` — which HOT files to evict first

We evict the files that are least likely to be needed again.
The heuristic: **lowest score first** (from Chapter 6's fuzzy scoring).
Files with no score get evicted before files that matched the current query.

We also always keep at least one HOT file — evicting everything defeats the purpose.

In [ ]:
COMPACT_THRESHOLD = 1.80   # trigger compaction when prompt > 80 % of budget
EVICT_TARGET      = 1.55   # compact down to 55 % of budget


def eviction_candidates(
    files:    list[dict],
    query:    str,
    n_keep:   int = 1,
) -> tuple[list[dict], list[dict]]:
    """
    Split *files* into (evict, keep).

    HOT files are sorted by their relevance score (ascending — lowest first).
    We evict all but the top *n_keep* HOT files.
    COLD files are never touched here.

    Returns (to_evict, to_keep) — both lists contain the original dicts.
    """
    hot  = [f for f in files if f.get("hot")]
    cold = [f for f in files if not f.get("hot")]

    if len(hot) <= n_keep:
        return [], files   # nothing to evict

    terms  = tokenize_query(query)
    # Sort HOT by score ascending (lowest relevance evicted first)
    scored = sorted(hot, key=lambda f: f.get("score", score_file(f, terms)))

    to_evict = scored[:-n_keep]    # all except the top n_keep
    to_keep  = scored[-n_keep:] + cold

    return to_evict, to_keep


# ── Quick check ──────────────────────────────────────────────────────────────
_demo_files = [
    {"path": "utils/parser.py",    "hot": True,  "score": 1.8,  "tokens": 420},
    {"path": "utils/formatter.py", "hot": True,  "score": 3.1,  "tokens": 380},
    {"path": "utils/validator.py", "hot": True,  "score": 1.3,  "tokens": 290},
    {"path": "main.py",            "hot": False, "score": 1.1,  "tokens": 120},
]
evict, keep = eviction_candidates(_demo_files, "how does the formatter work?")
print("Evict:", [f["path"] for f in evict])
print("Keep: ", [f["path"] for f in keep])

### 8.2 `summarise_file()` — compress one file with the LLM

We ask the model for a terse technical summary in plain prose — no code blocks,
no bullet points. The goal is a 3–5 sentence description that preserves
the key exported names and their purpose, so later queries can still
reference this file even though its full content is no longer in the prompt.

In [ ]:
def summarise_file(file_dict: dict) -> dict:
    """
    Ask the LLM to compress *file_dict["content"]* to a short summary.

    Returns a new dict with:
      - "content"  replaced by the summary text
      - "tokens"   updated to the summary's token count
      - "hot"      set to False  (evicted from active prompt)
      - "summary"  set to True   (flag so later code knows this is compressed)
    """
    content = file_dict.get("content", "")
    if not content.strip():
        return {**file_dict, "hot": False, "summary": True}

    prompt = (
        f"Summarise the following source file in 3–5 sentences of plain prose. "
        f"Name the key functions/classes and what they do. "
        f"No code blocks, no bullet points.\n\n"
        f"FILE: {file_dict['path']}\n\n{content}"
    )
    summary_text, _ = chat([{"role": "user", "content": prompt}])

    return {
        **file_dict,
        "content": f"[SUMMARY of {file_dict['path']}]\n{summary_text}",
        "tokens":  count_tokens(summary_text),
        "hot":     False,
        "summary": True,
    }


# ── Demo: summarise parser.py ────────────────────────────────────────────────
_parser_meta   = glob_files("parser.py", repo_root="sample_project")[0]
_parser_loaded = jit_read(_parser_meta, repo_root="sample_project")

print(f"Before: {_parser_loaded['tokens']} tokens")
_parser_summary = summarise_file(_parser_loaded)
print(f"After : {_parser_summary['tokens']} tokens  "
      f"({_parser_summary['tokens']/_parser_loaded['tokens']*100:.0f}% of original)\n")
print(_parser_summary["content"])

### 8.3 `compact()` — orchestrate the full compaction pass

`compact()` is called by the agent whenever the token count crosses
`COMPACT_THRESHOLD`. It loops through eviction candidates, summarises
each one, and returns the updated file list plus a log of what was freed.

In [ ]:
def compact(
    files:      list[dict],
    query:      str,
    token_used: int,
) -> tuple[list[dict], int, list[str]]:
    """
    If *token_used* exceeds COMPACT_THRESHOLD × TOKEN_BUDGET, evict and
    summarise the least-relevant HOT files until usage drops below
    EVICT_TARGET × TOKEN_BUDGET.

    Returns:
        (updated_files, new_token_used, log_lines)
    where log_lines is a human-readable list of what was compacted.
    """
    threshold = int(TOKEN_BUDGET * COMPACT_THRESHOLD)
    target    = int(TOKEN_BUDGET * EVICT_TARGET)

    if token_used <= threshold:
        return files, token_used, []   # nothing to do

    log     = [f"Compaction triggered: {token_used:,} / {TOKEN_BUDGET:,} tokens "
               f"({token_used/TOKEN_BUDGET*100:.0f}%)"]
    updated = list(files)
    used    = token_used

    while used > target:
        to_evict, _ = eviction_candidates(updated, query, n_keep=1)
        if not to_evict:
            break

        # Evict one file at a time
        victim = to_evict[0]
        before = victim.get("tokens", 0)

        summarised = summarise_file(victim)
        after      = summarised["tokens"]
        saved      = before - after

        # Replace the victim in the list
        updated = [summarised if f["path"] == victim["path"] else f
                   for f in updated]
        used   -= saved
        log.append(f"  compacted {victim['path']}: {before}t → {after}t  (saved {saved}t)")

    log.append(f"After compaction: {used:,} / {TOKEN_BUDGET:,} tokens "
               f"({used/TOKEN_BUDGET*100:.0f}%)")
    return updated, used, log

### 8.4 Simulation — watch compaction fire

We simulate a session that loads all sample files and artificially inflates
the token count past the compaction threshold, then watch `compact()` log
what it evicts and how much budget it recovers.

In [ ]:
query = "How does the formatter handle missing fields?"
repo  = "sample_project"

# Load all Python files HOT
all_files   = glob_files("*.py", repo_root=repo)
loaded      = [jit_read(f, repo_root=repo) for f in all_files]
# Give them fuzzy scores so eviction ordering works
terms       = tokenize_query(query)
loaded      = [{**f, "score": score_file(f, terms)} for f in loaded]
token_used  = sum(f["tokens"] for f in loaded)

# Artificially inflate to simulate a long session
# (pad to 85 % of budget so compaction fires)
pad_tokens  = int(TOKEN_BUDGET * 1.85) - token_used
token_used += max(pad_tokens, 0)

show_rule("Before compaction")
show_panel(
    query        = query,
    token_used   = token_used,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
    strategy     = "compact",
    prompt_size  = token_used,
)

# Run compaction
updated, new_used, log = compact(loaded, query, token_used)

show_rule("Compaction log")
for line in log:
    print(f"  {line}")

show_rule("After compaction")
show_panel(
    query        = query,
    token_used   = new_used,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in updated],
    strategy     = "compact",
    prompt_size  = new_used,
)

---
## Chapter 9 — Semantic RAG

**Goal:** retrieve files by meaning, not just keywords or filenames.

Grep and fuzzy scoring both rely on surface-level text overlap.
They miss cases like: *"where is the nesting logic handled?"* — the relevant
function is `_flatten` in `parser.py`, but neither the query nor the filename
contains the word "flatten".

Semantic retrieval fixes this by:
1. **Embedding** each file (or chunk) into a vector using `nomic-embed-text`
2. **Embedding** the query into the same vector space at runtime
3. **Ranking** files by cosine similarity — close vectors = similar meaning

The embeddings are computed once and cached in memory. On subsequent calls
only the (cheap) query embedding is recomputed.

**You will:**
- Write `embed()` — call Ollama's embedding endpoint
- Write `build_index()` — embed all files and store vectors in memory
- Write `semantic_retrieve()` — embed the query and return ranked files
- Compare semantic vs grep for a query that grep misses

### 9.1 Dependencies

Chapter 9 needs `numpy` for the cosine similarity calculation.
Make sure `nomic-embed-text` is pulled — this is the `OLLAMA_EMBED` model
set in the config cell at the top.

In [ ]:
import subprocess, sys

_CH8_DEPS = ["numpy"]
for _pkg in _CH8_DEPS:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", _pkg],
        stdout=subprocess.DEVNULL,
    )

import numpy as np
print("Chapter 9 dependencies ready:", _CH8_DEPS)

# Confirm the embed model is available
ok, models = ping_ollama()
if ok and any(OLLAMA_EMBED in m for m in models):
    print(f"Embed model '{OLLAMA_EMBED}' is available.")
else:
    print(f"WARNING: embed model '{OLLAMA_EMBED}' not found.")
    print(f"Run: ollama pull {OLLAMA_EMBED}")

### 9.2 `embed()` — call Ollama's embedding endpoint

Ollama exposes `/api/embed` (or `/api/embeddings` in older versions).
It accepts a model name and a string, returns a float vector.
We normalise the vector to unit length immediately so cosine similarity
later is just a dot product.

In [ ]:
def embed(text: str, model: str = OLLAMA_EMBED) -> np.ndarray:
    """
    Return a unit-normalised embedding vector for *text*.

    Uses Ollama's /api/embed endpoint (introduced in Ollama 1.1.26).
    Falls back to /api/embeddings for older installations.
    """
    payload = {"model": model, "input": text}
    try:
        r = requests.post(f"{OLLAMA_BASE_URL}/api/embed",
                          json=payload, timeout=60)
        r.raise_for_status()
        vec = r.json()["embeddings"][0]
    except Exception:
        # Fallback for older Ollama versions
        payload_old = {"model": model, "prompt": text}
        r = requests.post(f"{OLLAMA_BASE_URL}/api/embeddings",
                          json=payload_old, timeout=60)
        r.raise_for_status()
        vec = r.json()["embedding"]

    arr  = np.array(vec, dtype=np.float32)
    norm = np.linalg.norm(arr)
    return arr / norm if norm > 0 else arr


# Quick smoke test
_v = embed("hello world")
print(f"Vector dimension : {len(_v)}")
print(f"Vector norm      : {np.linalg.norm(_v):.6f}  (should be ~2.0)")

### 9.3 `build_index()` — embed every file, cache in memory

We embed each file's content and store the vector alongside the file metadata.
This is the expensive step — one embedding call per file.
After the first call the index lives in the `_EMBED_INDEX` dict and is reused.

In [ ]:
# In-memory index: repo_root → list of {"path", "vector", "bytes", "hot"}
_EMBED_INDEX: dict[str, list[dict]] = {}


def build_index(
    repo_root:  str = REPO_ROOT,
    extensions: set = CODE_EXTENSIONS,
    force:      bool = REBUILD_INDEX,
) -> list[dict]:
    """
    Embed every source file under *repo_root* and cache the result.

    Returns the index (list of dicts with a "vector" key).
    On subsequent calls the cached index is returned unless *force=True*.
    """
    if repo_root in _EMBED_INDEX and not force:
        return _EMBED_INDEX[repo_root]

    files = glob_files("*", repo_root=repo_root)
    files = [f for f in files
             if Path(f["path"]).suffix.lower() in extensions]

    index = []
    for f in files:
        full_path = Path(repo_root) / f["path"]
        try:
            content = full_path.read_text(errors="ignore")
        except OSError:
            continue

        if not content.strip():
            # Skip empty files — embed() returns a 0-dim vector for empty
            # strings which breaks np.dot later.
            continue

        # Truncate very long files before embedding to avoid timeouts
        snippet = content[:4000]
        vec = embed(snippet)

        if len(vec) == 0:
            # Embedding model returned nothing (empty string edge case)
            continue

        index.append({**f, "vector": vec, "content": content,
                      "tokens": count_tokens(content)})
        print(f"  embedded  {f['path']}")

    _EMBED_INDEX[repo_root] = index
    return index


show_rule("Building semantic index for sample_project")
idx = build_index(repo_root="sample_project")
print(f"\nIndex contains {len(idx)} files.")


### 9.4 `semantic_retrieve()` — rank by cosine similarity

Cosine similarity between unit vectors is just a dot product.
We embed the query, dot it against every file vector, and sort descending.
The result dict shape matches what `budget_load()` and `show_panel()` expect.

In [ ]:
def semantic_retrieve(
    query:     str,
    repo_root: str = REPO_ROOT,
    top_k:     int = 5,
) -> list[dict]:
    """
    Return the top-*k* files from the semantic index, ranked by cosine
    similarity to *query*.

    Each returned dict has a "score" key (cosine similarity, 0–1).
    """
    index     = build_index(repo_root=repo_root)
    query_vec = embed(query)

    if len(query_vec) == 0:
        # Query embedding failed (e.g. empty query) — fall back to empty list
        return []

    scored = []
    for entry in index:
        vec = entry["vector"]
        if len(vec) == 0 or vec.shape != query_vec.shape:
            # Stale index entry with a bad vector — skip rather than crash
            continue
        sim = float(np.dot(query_vec, vec))
        scored.append({
            "path":    entry["path"],
            "bytes":   entry["bytes"],
            "tokens":  entry["tokens"],
            "content": entry["content"],
            "hot":     False,
            "score":   round(sim, 4),
        })

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]


# ── Demo: the query grep struggles with ──────────────────────────────────────
query = "Where is the nesting logic handled?"

sem_results  = semantic_retrieve(query, repo_root="sample_project")
grep_results = grep_rank(query, repo_root="sample_project")

tbl = Table(title="Semantic vs Grep", header_style="bold cyan")
tbl.add_column("Method")
tbl.add_column("Rank", justify="right")
tbl.add_column("Score", justify="right")
tbl.add_column("File")

for i, f in enumerate(sem_results, 1):
    tbl.add_row("semantic", str(i), f"[green]{f['score']}[/green]", f["path"])

for i, f in enumerate(grep_results, 1):
    tbl.add_row("grep",     str(i), f"[yellow]{f['score']}[/yellow]", f["path"])

print(tbl)


### 9.5 Try it end-to-end

Ask the nesting query. Semantic retrieval finds `parser.py` at the top
because `_flatten` is semantically close to "nesting logic",
even though neither word appears in the other.

In [ ]:
query = "Where is the nesting logic handled?"
repo  = "sample_project"

candidates   = semantic_retrieve(query, repo_root=repo)
manifest     = load_manifest(repo)
manifest_tok = manifest["tokens"]
loaded_files = budget_load(candidates, already_used=manifest_tok, repo_root=repo)

hot_files   = [f for f in loaded_files if f["hot"]]
prompt_size = manifest_tok + sum(f["tokens"] for f in hot_files) + count_tokens(query)

show_rule("Chapter 9 — semantic retrieval before LLM call")
show_panel(
    query        = query,
    token_used   = prompt_size,
    files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy     = "semantic RAG",
    prompt_size  = prompt_size,
)

reply, tokens_used = ask_with_manifest(query, repo_root=repo, files=hot_files)

show_rule("After call")
show_panel(
    query       = query,
    token_used  = tokens_used,
    files       = [{"path": f["path"], "hot": f["hot"]} for f in loaded_files],
    strategy    = "semantic RAG",
    prompt_size = prompt_size,
)
show_reply(reply)


---
## Chapter 10 — Full Pipeline

**Goal:** one function call, `run(query, repo)`, does everything.

Every chapter so far introduced one tool. Chapter 10 wires them together:

```
query
  │
  ├─ manifest loaded (always)
  │
  ├─ strategy picker
  │     ├─ "semantic"  → semantic_retrieve()
  │     ├─ "grep"      → grep_rank()
  │     └─ "fuzzy"     → rank_files(glob_files())
  │
  ├─ budget_load()   — JIT read, HOT/COLD split
  │
  ├─ compact()       — if > COMPACT_THRESHOLD
  │
  └─ ask_with_manifest()  → reply
```

The **strategy picker** is a small classifier that looks at the query:
- Contains a specific symbol or quoted string → grep (looking for code)
- Vague / conceptual phrasing → semantic (understanding meaning)
- Everything else → fuzzy (fast, good enough for named-file queries)

**You will:**
- Write `pick_strategy()` — classify the query
- Write `run()` — the full pipeline in one call
- Try three queries that each route to a different strategy

### 10.1 `pick_strategy()` — classify the query

A lightweight heuristic — no ML needed.
The three signals are mutually exclusive and checked in priority order.

In [ ]:
# Words that suggest the user is asking about a concept, not looking for a symbol
_CONCEPTUAL_WORDS = {
    "why", "how", "explain", "what", "design", "architecture",
    "pattern", "approach", "concept", "idea", "strategy", "logic",
    "purpose", "reason", "difference", "relationship",
}

# Patterns that suggest the user is looking for a specific symbol or string
_GREP_SIGNALS = re.compile(
    r'(?:'
    r'"[^"]+"'              # quoted string
    r'|`[^`]+`'             # backtick symbol
    r'|raise\s+\w+'         # raise SomeError
    r'|def\s+\w+'           # def function_name
    r'|class\s+\w+'         # class ClassName
    r'|import\s+\w+'        # import something
    r'|\b[A-Z][a-zA-Z]+Error\b'  # ExceptionName
    r')'
)


def pick_strategy(query: str) -> str:
    """
    Classify *query* into one of three retrieval strategies:
      "grep"     — looking for a specific symbol / string in code
      "semantic" — conceptual / meaning-based question
      "fuzzy"    — everything else (filename-level search)
    """
    # Grep signal: backticks, quotes, error names, def/class/import
    if _GREP_SIGNALS.search(query):
        return "grep"

    # Semantic signal: conceptual vocabulary
    words = set(query.lower().split())
    if words & _CONCEPTUAL_WORDS:
        return "semantic"

    return "fuzzy"


# ── Check the three example queries ──────────────────────────────────────────
for q in [
    "Where does the code raise a `TypeError`?",
    "Why does the parser wrap scalars in a dict?",
    "How does the formatter handle missing fields?",
]:
    print(f"  {pick_strategy(q):8s}  {q}")

### 10.2 `run()` — the full pipeline

This is the function all later chapters call.
It returns a `RunResult` named tuple so callers can inspect
the files that were loaded, the strategy used, and the token counts.

In [ ]:
from typing import NamedTuple


class RunResult(NamedTuple):
    reply:       str
    strategy:    str
    files:       list[dict]   # full list, HOT and COLD
    tokens_used: int
    compact_log: list[str]    # empty if compaction didn't fire


def run(
    query:     str,
    repo_root: str  = REPO_ROOT,
    strategy:  str  = "auto",   # "auto" | "fuzzy" | "grep" | "semantic"
    top_k:     int  = 8,
) -> RunResult:
    """
    Full retrieval + generation pipeline.

    Parameters
    ----------
    query     : the user's question
    repo_root : path to the repository to search
    strategy  : retrieval strategy; "auto" lets pick_strategy() decide
    top_k     : max candidates to consider before budget_load()
    """
    # ── 1. Manifest ─────────────────────────────────────────────────────────
    manifest     = load_manifest(repo_root)
    manifest_tok = manifest["tokens"]

    # ── 2. Strategy selection ────────────────────────────────────────────────
    strat = pick_strategy(query) if strategy == "auto" else strategy

    # ── 3. Retrieval ─────────────────────────────────────────────────────────
    if strat == "grep":
        candidates = grep_rank(query, repo_root=repo_root)
        # Append fuzzy extras for files with zero grep hits
        found_paths = {f["path"] for f in candidates}
        extras = [f for f in rank_files(glob_files("*.py", repo_root=repo_root), query)
                  if f["path"] not in found_paths]
        candidates = (candidates + extras)[:top_k]

    elif strat == "semantic":
        candidates = semantic_retrieve(query, repo_root=repo_root, top_k=top_k)

    else:   # fuzzy
        candidates = rank_files(glob_files("*.py", repo_root=repo_root), query)[:top_k]

    # ── 4. Budget-aware JIT load ─────────────────────────────────────────────
    loaded = budget_load(candidates, already_used=manifest_tok, repo_root=repo_root)

    # ── 5. Compaction (if needed) ─────────────────────────────────────────────
    hot_tok = sum(f.get("tokens", 0) for f in loaded if f["hot"])
    total   = manifest_tok + hot_tok + count_tokens(query)
    loaded, total, compact_log = compact(loaded, query, total)

    # ── 6. Show panel ─────────────────────────────────────────────────────────
    hot_files = [f for f in loaded if f["hot"]]
    show_panel(
        query        = query,
        token_used   = total,
        files        = [{"path": f["path"], "hot": f["hot"]} for f in loaded],
        strategy     = strat,
        prompt_size  = total,
    )

    # ── 7. Generate reply ─────────────────────────────────────────────────────
    reply, tokens_used = ask_with_manifest(query, repo_root=repo_root, files=hot_files)

    return RunResult(
        reply        = reply,
        strategy     = strat,
        files        = loaded,
        tokens_used  = tokens_used,
        compact_log  = compact_log,
    )

### 10.3 Try all three strategies

Three queries, each routed automatically to a different strategy.
Watch the panel's **Strategy** field change on each run.

In [ ]:
repo = "sample_project"

queries = [
    "Where does the code raise a `TypeError`?",           # → grep
    "Why does the parser wrap scalars in a dict?",        # → semantic
    "What does the formatter do with the delimiter?",     # → fuzzy
]

for q in queries:
    show_rule(f"query: {q}")
    result = run(q, repo_root=repo)
    print(f"\nStrategy used: {result.strategy}")
    show_reply(result.reply)


---
## Chapter 11 — Write + Diff

**Goal:** give the agent the ability to modify files on disk.

Up to now the agent only reads. Chapter 11 adds the write side:
the agent proposes a change, we show a coloured unified diff,
and only write if confirmed (or if `dry_run=False` is set explicitly).

Two principles guide this chapter:

1. **Show before you write.** The diff is always printed before any file is touched.
2. **Patch, don't replace.** The agent receives the original file content
   and returns the complete new content. We generate the diff from those two
   strings — no fragile line-number arithmetic.

**You will:**
- Write `make_diff()` — unified diff between two strings, with colour
- Write `apply_patch()` — ask the LLM to rewrite a file given an instruction
- Write `write_file()` — diff + write, with dry-run support
- Try it: ask the agent to add a docstring to a function

### 11.1 `make_diff()` — coloured unified diff

Python's `difflib.unified_diff` does the heavy lifting.
`print_diff()` renders the result as colour-coded HTML in the notebook:
green for additions, red for removals, blue for chunk headers.


In [ ]:
import difflib


def make_diff(
    original:  str,
    proposed:  str,
    file_path: str = "<file>",
    context:   int = 3,
) -> str:
    """
    Return a unified diff string between *original* and *proposed*.
    Returns an empty string if they are identical.
    """
    orig_lines = original.splitlines(keepends=True)
    new_lines  = proposed.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(
        orig_lines, new_lines,
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
        n=context,
    ))
    return "".join(diff_lines)


def print_diff(diff: str) -> None:
    """Render a unified diff with colour-coded lines in the notebook."""
    if not diff:
        print("No changes.")
        return
    lines = diff.splitlines()
    html_lines = []
    for ln in lines:
        safe = ln.replace("&","&amp;").replace("<","&lt;")
        if ln.startswith("+") and not ln.startswith("+++"):
            html_lines.append(f'<span style="color:#2da44e">{safe}</span>')
        elif ln.startswith("-") and not ln.startswith("---"):
            html_lines.append(f'<span style="color:#cf222e">{safe}</span>')
        elif ln.startswith("@@"):
            html_lines.append(f'<span style="color:#0969da">{safe}</span>')
        else:
            html_lines.append(f'<span style="color:#57606a">{safe}</span>')
    display(HTML(
        '<pre style="background:#f6f8fa;border:1px solid #d0d7de;border-radius:6px;'
        'padding:12px;font-size:0.82rem;overflow-x:auto;line-height:1.4">'
        + "\n".join(html_lines) + "</pre>"
    ))


# ── Quick demo ────────────────────────────────────────────────────────────────
_original = 'def add(a, b):\n    return a + b\n'
_proposed = 'def add(a: int, b: int) -> int:\n    """Return the sum of a and b."""\n    return a + b\n'
print_diff(make_diff(_original, _proposed, "math_utils.py"))

### 11.2 `apply_patch()` — ask the LLM to rewrite a file

We give the model the original file content and a plain-English instruction.
It returns the **complete** new file — not a patch, not just the changed lines.
This is deliberate: asking the model to produce a full file is more reliable
than asking it to produce a diff, which it frequently gets wrong.

In [ ]:
def apply_patch(
    file_path:   str,
    instruction: str,
    repo_root:   str = REPO_ROOT,
) -> tuple[str, str]:
    """
    Ask the LLM to apply *instruction* to the file at *file_path*.

    Returns (original_content, proposed_content).
    The proposed content is the raw LLM output, stripped of markdown fences.
    If the file does not exist yet, original is treated as empty string
    (allows the agent to create new files).
    """
    # Normalise: strip repo_root prefix if the model included it
    rel = file_path
    for prefix in (repo_root + "/", repo_root + os.sep):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]
            break

    full_path = Path(repo_root) / rel
    original  = full_path.read_text(errors="ignore") if full_path.exists() else ""

    prompt = (
        f"Apply the following instruction to the source file below.\n"
        f"Return ONLY the complete modified file — no explanations, "
        f"no markdown fences, no commentary.\n\n"
        f"INSTRUCTION: {instruction}\n\n"
        f"FILE: {rel}\n"
        f"```\n{original}\n```"
    )

    proposed_raw, _ = chat_continued([{"role": "user", "content": prompt}])

    # Strip markdown code fences if the model added them anyway
    proposed = re.sub(r"^```[a-zA-Z]*\n?", "", proposed_raw.strip())
    proposed = re.sub(r"\n?```$", "", proposed)

    return original, proposed.strip() + "\n"

### 11.3 `write_file()` — diff, confirm, write

`write_file()` wraps `apply_patch()`:
- Always shows the diff first
- In `dry_run=True` mode (default) it stops there — nothing is written
- In `dry_run=False` mode it writes the file after showing the diff

In [ ]:
def write_file(
    file_path:   str,
    instruction: str,
    repo_root:   str  = REPO_ROOT,
    dry_run:     bool = True,
) -> tuple[str, str, str]:
    """
    Apply *instruction* to *file_path*, show the diff, optionally write.

    Parameters
    ----------
    file_path   : path relative to repo_root
    instruction : plain-English change request
    repo_root   : repository root
    dry_run     : if True, print the diff but do NOT write to disk

    Returns (original, proposed, diff_text).
    """
    print(f"\nApplying: {instruction}")
    print(f"File: {file_path}\n")

    original, proposed = apply_patch(file_path, instruction, repo_root)
    diff = make_diff(original, proposed, file_path)

    show_rule("Proposed diff")
    print_diff(diff)

    if not diff:
        print("No changes proposed by the model.")
        return original, proposed, diff

    if dry_run:
        print("\nDRY RUN — file not written. Set dry_run=False to apply.")
    else:
        full_path = Path(repo_root) / file_path
        full_path.write_text(proposed)
        print(f"\n✓ Written: {full_path}")

    return original, proposed, diff

### 11.4 Try it — add type hints to `_flatten`

We run in `dry_run=True` first so you can inspect the diff.
Change it to `dry_run=False` to actually write the file.

In [ ]:
original, proposed, diff = write_file(
    file_path   = "utils/parser.py",
    instruction = "Add PEP 484 type annotations to all function signatures. "
                  "Do not change any logic or add any comments.",
    repo_root   = "sample_project",
    dry_run     = True,    # ← change to False to write to disk
)

---
## Chapter 12 — Agent Loop

**Goal:** turn the agent from a question-answerer into an autonomous task executor.

So far each chapter required the user to drive every step.
Chapter 12 adds a loop that:

1. **Plans** — asks the LLM to break the task into ordered steps
2. **Executes** — for each step, decides whether to read (→ `run()`) or write (→ `write_file()`)
3. **Verifies** — after writing, asks the LLM to confirm the change makes sense
4. **Iterates** — repeats until the plan is complete or `max_steps` is reached

The loop is intentionally simple — no tool-calling API, no JSON schema enforcement.
The LLM outputs plain text; the loop parses a lightweight convention:

```
STEP: <description>
ACTION: read | write
TARGET: <file path>   (for write actions)
INSTRUCTION: <what to change>   (for write actions)
```

**You will:**
- Write `plan_task()` — ask the LLM to emit a structured step list
- Write `execute_step()` — dispatch one step to read or write
- Write `agent_loop()` — run the full plan with a step cap and verification

### 12.1 `plan_task()` — break a task into steps

We give the model the manifest and a system prompt that enforces
the `STEP / ACTION / TARGET / INSTRUCTION` format.
The output is parsed into a list of step dicts.

In [ ]:
import json

_PLAN_SYSTEM = """\
You are a coding agent. Given a task and a project map, produce an ordered
list of steps to complete the task.

Each step must follow this EXACT format (one step per block, blank line between):

STEP: <one-sentence description>
ACTION: read
TARGET: <question to answer about the codebase>

or

STEP: <one-sentence description>
ACTION: write
TARGET: <file path relative to repo root>
INSTRUCTION: <precise instruction for what to change in that file>

or

STEP: <one-sentence description>
ACTION: tool
TOOL: <one of: bash, glob, grep, read, edit, write, notebookedit, todowrite, websearch, skill, enterworktree, exitworktree, croncreate, crondelete, cronlist>
ARGS: <JSON object of arguments for the tool>

Rules:
- Use ACTION: read to gather information before writing.
- Use ACTION: write only when you are confident what to change.
- Use ACTION: tool when a direct tool invocation is required.
- Be specific in INSTRUCTION — the change will be applied by another model with no extra context.
- Emit at most 6 steps.
- Do not add any text outside the step blocks.
"""


def plan_task(task: str, repo_root: str = REPO_ROOT) -> list[dict]:
    """
    Ask the LLM to produce a step-by-step plan for *task*.

    Returns a list of dicts:
        {"step": str, "action": "read"|"write"|"tool",
         "target": str, "instruction": str, "tool": str, "args": dict}
    """
    manifest = load_manifest(repo_root)
    prompt   = (
        f"PROJECT MAP:\n{manifest['text']}\n\n"
        f"TASK: {task}"
    )

    raw, _ = chat_continued([
        {"role": "system",  "content": _PLAN_SYSTEM},
        {"role": "user",    "content": prompt},
    ])

    # Parse the step blocks
    steps = []
    for block in re.split(r"\n{2,}", raw.strip()):
        lines = {
            m.group(1).lower(): m.group(2).strip()
            for line in block.splitlines()
            if (m := re.match(r"^(STEP|ACTION|TARGET|INSTRUCTION|TOOL|ARGS):\s*(.+)$",
                               line, re.IGNORECASE))
        }
        if "action" in lines and ("target" in lines or "tool" in lines):
            args = {}
            if "args" in lines:
                try:
                    args = json.loads(lines['args'])
                except Exception:
                    args = {"_raw": lines['args']}
            steps.append({
                "step":        lines.get("step", ""),
                "action":      lines["action"].lower(),
                "target":      lines.get("target", ""),
                "instruction": lines.get("instruction", ""),
                "tool":        lines.get("tool", "").lower(),
                "args":        args,
            })

    return steps


# ── Preview a plan ────────────────────────────────────────────────────────────
task  = "Add type annotations to all functions in utils/ and update their docstrings."
plan  = plan_task(task, repo_root="sample_project")

show_rule("Plan")
for i, s in enumerate(plan, 1):
    action_color = {"write": "🟢", "read": "🔵", "tool": "🟡"}.get(s["action"], "⚪")
    print(f"{i}. {action_color} {s['action'].upper()}  {s['step']}")
    if s['action'] == "tool":
        print(f"   tool:   {s['tool']}")
        print(f"   args:   {s['args']}")
    else:
        print(f"   target: {s['target']}")
    if s['instruction']:
        print(f"   instr:  {s['instruction'][:80]}…")
    print()


### 12.2 `execute_step()` — dispatch one step

A thin dispatcher: read steps go to `run()`, write steps go to `write_file()`.

In [ ]:
import json
import subprocess
from pathlib import Path

import requests

CURRENT_REPO_ROOT = REPO_ROOT
_WORKTREE_STACK = []


def _resolve_root(repo_root: str | None = None) -> str:
    return repo_root or CURRENT_REPO_ROOT


def tool_bash(cmd: str, repo_root: str | None = None) -> dict:
    root = _resolve_root(repo_root)
    result = subprocess.run(cmd, shell=True, cwd=root, capture_output=True, text=True)
    return {
        "cmd": cmd,
        "cwd": root,
        "code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


def tool_glob(pattern: str, repo_root: str | None = None) -> list[dict]:
    return glob_files(pattern, repo_root=_resolve_root(repo_root))


def tool_grep(pattern: str, repo_root: str | None = None) -> list[dict]:
    return grep_repo(pattern, repo_root=_resolve_root(repo_root))


def tool_read(file_path: str, repo_root: str | None = None) -> dict:
    root = _resolve_root(repo_root)
    full = Path(root) / file_path
    if not full.exists():
        return {"error": "not found", "path": str(full)}
    return {"path": str(full), "content": full.read_text(errors="ignore")}


def tool_write(
    file_path: str,
    content: str,
    repo_root: str | None = None,
    append: bool = False,
) -> dict:
    root = _resolve_root(repo_root)
    full = Path(root) / file_path
    full.parent.mkdir(parents=True, exist_ok=True)
    mode = "a" if append else "w"
    with open(full, mode) as f:
        f.write(content)
    return {"path": str(full), "bytes": len(content)}


def tool_edit(
    file_path: str,
    instruction: str,
    repo_root: str | None = None,
    dry_run: bool = False,
) -> dict:
    original, proposed, diff = write_file(
        file_path=file_path,
        instruction=instruction,
        repo_root=_resolve_root(repo_root),
        dry_run=dry_run,
    )
    return {"original": original, "proposed": proposed, "diff": diff}


def tool_notebookedit(
    path: str,
    cell_index: int,
    new_source: str | list[str],
    repo_root: str | None = None,
) -> dict:
    root = _resolve_root(repo_root)
    nb_path = Path(root) / path
    nb = json.loads(nb_path.read_text())
    cell = nb["cells"][cell_index]
    if cell.get("cell_type") != "code":
        return {"error": "cell is not code", "cell_type": cell.get("cell_type")}
    if isinstance(new_source, str):
        cell["source"] = [line if line.endswith("\n") else line + "\n" for line in new_source.splitlines()]
    else:
        cell["source"] = new_source
    nb_path.write_text(json.dumps(nb, ensure_ascii=False, indent=1))
    return {"path": str(nb_path), "cell_index": cell_index}



def tool_websearch(query: str, max_results: int = 5, repo_root: str | None = None) -> list[dict]:
    _ = repo_root
    r = requests.get(
        "https://duckduckgo.com/html/",
        params={"q": query},
        headers={"User-Agent": "pocket-agent"},
        timeout=10,
    )
    html = r.text
    results = []
    for m in re.finditer(r'class="result__a"[^>]*href="(.*?)"[^>]*>(.*?)</a>', html):
        url = m.group(1)
        title = re.sub(r"<.*?>", "", m.group(2))
        results.append({"title": title, "url": url})
        if len(results) >= max_results:
            break
    return results


def tool_todowrite(items: str | list[str], repo_root: str | None = None, mode: str = "append") -> dict:
    root = _resolve_root(repo_root)
    path = Path(root) / ".pocket_agent_todo.md"
    if isinstance(items, str):
        items = [items]
    text = "\n".join(f"- [ ] {i}" for i in items) + "\n"
    if mode == "replace":
        path.write_text(text)
    else:
        with open(path, "a") as f:
            f.write(text)
    return {"path": str(path), "count": len(items)}


def tool_skill(repo_root: str | None = None) -> dict:
    _ = repo_root
    return {"tools": sorted(TOOLS.keys())}


def tool_enterworktree(path: str, repo_root: str | None = None) -> dict:
    global CURRENT_REPO_ROOT
    root = _resolve_root(repo_root)
    _WORKTREE_STACK.append(root)
    CURRENT_REPO_ROOT = path
    return {"current": CURRENT_REPO_ROOT, "stack": list(_WORKTREE_STACK)}


def tool_exitworktree(repo_root: str | None = None) -> dict:
    global CURRENT_REPO_ROOT
    _ = repo_root
    if _WORKTREE_STACK:
        CURRENT_REPO_ROOT = _WORKTREE_STACK.pop()
    return {"current": CURRENT_REPO_ROOT, "stack": list(_WORKTREE_STACK)}


def _cron_path(repo_root: str | None = None) -> Path:
    root = _resolve_root(repo_root)
    return Path(root) / ".pocket_agent_cron.json"


def _load_cron(repo_root: str | None = None) -> list[dict]:
    path = _cron_path(repo_root)
    if not path.exists():
        return []
    return json.loads(path.read_text())


def _save_cron(entries: list[dict], repo_root: str | None = None) -> None:
    path = _cron_path(repo_root)
    path.write_text(json.dumps(entries, ensure_ascii=False, indent=2))


def tool_croncreate(name: str, schedule: str, command: str, repo_root: str | None = None) -> dict:
    entries = _load_cron(repo_root)
    entry = {"name": name, "schedule": schedule, "command": command}
    entries.append(entry)
    _save_cron(entries, repo_root)
    return entry


def tool_cronlist(repo_root: str | None = None) -> list[dict]:
    return _load_cron(repo_root)


def tool_crondelete(name: str, repo_root: str | None = None) -> dict:
    entries = _load_cron(repo_root)
    kept = [e for e in entries if e.get("name") != name]
    _save_cron(kept, repo_root)
    return {"deleted": len(entries) - len(kept)}


TOOLS = {
    "bash": tool_bash,
    "glob": tool_glob,
    "grep": tool_grep,
    "read": tool_read,
    "edit": tool_edit,
    "write": tool_write,
    "notebookedit": tool_notebookedit,
    "todowrite": tool_todowrite,
    "websearch": tool_websearch,
    "skill": tool_skill,
    "enterworktree": tool_enterworktree,
    "exitworktree": tool_exitworktree,
    "croncreate": tool_croncreate,
    "crondelete": tool_crondelete,
    "cronlist": tool_cronlist,
}


def call_tool(name: str, repo_root: str | None = None, **kwargs) -> dict | list:
    key = name.lower()
    if key not in TOOLS:
        return {"error": f"unknown tool: {name}", "available": sorted(TOOLS.keys())}
    try:
        return TOOLS[key](repo_root=repo_root, **kwargs)
    except TypeError:
        # tool may not accept repo_root; retry without it
        return TOOLS[key](**kwargs)


def execute_step(
    step:      dict,
    repo_root: str  = REPO_ROOT,
    dry_run:   bool = True,
) -> str:
    """
    Execute one plan step.  Returns a short status string for the loop log.

    read  steps → run(target_as_query)
    write steps → write_file(target, instruction)
    tool  steps → call_tool(tool, **args)
    """
    action = step.get("action", "read")

    if action == "write":
        _, _, diff = write_file(
            file_path   = step["target"],
            instruction = step["instruction"],
            repo_root   = repo_root,
            dry_run     = dry_run,
        )
        return "wrote (dry)" if dry_run else "wrote"

    if action == "tool":
        result = call_tool(step.get("tool", ""), repo_root=repo_root, **step.get("args", {}))
        tool_result = str(result)
        suffix = "..." if len(tool_result) > 800 else ""
        print(f"\nTool result ({step.get('tool','')}):\n{tool_result[:800]}{suffix}\n")
        return f"tool:{step.get('tool','')}"

    else:   # read
        result = run(step["target"], repo_root=repo_root)
        suffix = "..." if len(result.reply) > 400 else ""
        print(f"\nRead result:\n{result.reply[:400]}{suffix}\n")
        return "read"


### 12.3 `agent_loop()` — plan → execute → verify

The loop runs the full plan, printing progress at each step.
After all write steps complete, it runs a final verification query
to ask the model whether the changes look correct.

In [ ]:
def agent_loop(
    task:      str,
    repo_root: str  = REPO_ROOT,
    dry_run:   bool = True,
    max_steps: int  = 8,
) -> list[dict]:
    """
    Autonomous task execution loop.

    1. Plan the task with plan_task()
    2. Execute each step with execute_step()
    3. Verify the result with a final read

    Returns the step log (list of dicts with "step", "action", "status").
    """
    show_rule(f"Agent Loop  ·  {task[:60]}")
    print(f"repo: {repo_root}  |  dry_run={dry_run}  |  max_steps={max_steps}\n")

    # ── 1. Plan ───────────────────────────────────────────────────────────────
    plan = plan_task(task, repo_root=repo_root)
    if not plan:
        print("Planning failed — no steps parsed from model output.")
        return []

    print(f"Plan: {len(plan)} step(s)\n")

    # ── 2. Execute ────────────────────────────────────────────────────────────
    log = []
    for i, step in enumerate(plan[:max_steps], 1):
        show_rule(f"Step {i}/{min(len(plan), max_steps)}  {step['action'].upper()}  {step['step']}")

        status = execute_step(step, repo_root=repo_root, dry_run=dry_run)
        log.append({**step, "status": status})
        print(f"↳ {status}")

    # ── 3. Verify ─────────────────────────────────────────────────────────────
    written = [s for s in log if s["action"] == "write"]
    if written:
        show_rule("Verification")
        verify_query = (
            f"Review whether the following task has been completed correctly: {task}\n"
            f"Files modified: {', '.join(s['target'] for s in written)}"
        )
        verification = run(verify_query, repo_root=repo_root)
        print(f"\nVerification result:\n{verification.reply}")

    show_rule("Loop complete")
    return log

### 12.4 Try it

Run the agent loop on a real task.
`dry_run=True` means no files are touched — you can set it to `False`
once you've reviewed the diffs and are happy with the plan.

In [ ]:
log = agent_loop(
    task      = "Add type annotations to all functions in utils/ and update their docstrings.",
    repo_root = "sample_project",
    dry_run   = True,    # ← change to False to write files
    max_steps = 6,
)

---
## Chapter 13 — Test Generation

**Goal:** the agent writes its own tests, runs them, and fixes failures.

This is the most complete demonstration of the full system.
Every function built across chapters 1–11 is in play:

- `run()` reads the source file to understand it (Ch 9)
- `write_file()` writes the test file to disk (Ch 10)
- `agent_loop()` orchestrates the plan (Ch 11)
- A subprocess call runs `pytest` and captures the output
- If tests fail, the failure output is fed back to `apply_patch()` and retried

**You will:**
- Write `generate_tests()` — ask the LLM to produce a pytest file for a module
- Write `run_tests()` — execute pytest, capture pass/fail/error
- Write `test_loop()` — generate → run → fix loop with a retry cap
- Try it: generate tests for `utils/formatter.py` and run them

### 13.1 `generate_tests()` — produce a pytest file

We read the source module first via `run()`, then ask the model
to write a complete `pytest` test file for it.
The test file is returned as a string — not written yet.

In [ ]:
def generate_tests(
    source_path: str,
    repo_root:   str = REPO_ROOT,
) -> str:
    """
    Generate a pytest test file for the module at *source_path*.

    Returns the test file content as a string (not written to disk yet).
    """
    full_path = Path(repo_root) / source_path
    source    = full_path.read_text(errors="ignore")

    prompt = (
        f"Write a complete pytest test file for the following Python module.\n\n"
        f"Requirements:\n"
        f"- Use pytest (not unittest)\n"
        f"- Cover the main happy path and at least two edge cases per function\n"
        f"- Import from the module using its dotted path relative to the repo root\n"
        f"  (e.g. 'from utils.formatter import to_csv')\n"
        f"- Return ONLY the test file — no explanation, no markdown fences\n\n"
        f"SOURCE FILE: {source_path}\n\n{source}"
    )

    raw, _ = chat_continued([{"role": "user", "content": prompt}])

    # Strip fences if model added them
    code = re.sub(r"^```[a-zA-Z]*\n?", "", raw.strip())
    code = re.sub(r"\n?```$", "", code)
    return code.strip() + "\n"


# ── Preview without writing ───────────────────────────────────────────────────
show_rule("Generating tests for utils/formatter.py")
test_code = generate_tests("utils/formatter.py", repo_root="sample_project")
display(Markdown(f"```python\n{test_code}\n```"))


### 13.2 `run_tests()` — execute pytest, capture results

We run `pytest` in a subprocess and return a structured result:
passed, failed, error count, and the raw output for feeding back
to the model if something went wrong.

In [ ]:
import subprocess as _sp


def run_tests(
    test_path: str,
    repo_root: str = REPO_ROOT,
) -> dict:
    """
    Run pytest on *test_path* (relative to repo_root).

    Returns:
        {
          "passed":  int,
          "failed":  int,
          "errors":  int,
          "output":  str,   # full pytest stdout+stderr
          "ok":      bool,  # True if passed > 0 and failed == errors == 0
        }
    """
    result = _sp.run(
        [sys.executable, "-m", "pytest", test_path, "-v", "--tb=short",
         "--no-header", "-q"],
        capture_output=True,
        text=True,
        cwd=repo_root,   # run from inside repo_root, so test_path is relative
    )

    output = result.stdout + result.stderr

    # Parse summary line: "3 passed, 1 failed, 0 errors"
    passed = int(m.group(1)) if (m := re.search(r"(\d+) passed",  output)) else 0
    failed = int(m.group(1)) if (m := re.search(r"(\d+) failed",  output)) else 0
    errors = int(m.group(1)) if (m := re.search(r"(\d+) error",   output)) else 0

    return {
        "passed": passed,
        "failed": failed,
        "errors": errors,
        "output": output,
        "ok":     passed > 0 and failed == 0 and errors == 0,
    }

### 13.3 `test_loop()` — generate → run → fix → retry

The loop:
1. Generate a test file
2. Write it to disk
3. Run pytest
4. If tests fail, show the failure output, ask the model to fix, overwrite, repeat
5. Stop when all tests pass or `max_retries` is exhausted

In [ ]:
def test_loop(
    source_path: str,
    repo_root:   str = REPO_ROOT,
    max_retries: int = 3,
) -> dict:
    """
    Full generate → run → fix loop for *source_path*.

    Returns the final run_tests() result dict plus a "attempts" key.
    """
    # Derive test file path: utils/formatter.py → tests/test_formatter_gen.py
    stem      = Path(source_path).stem
    test_path = f"tests/test_{stem}_gen.py"
    full_test = Path(repo_root) / test_path
    full_test.parent.mkdir(parents=True, exist_ok=True)

    show_rule(f"Test Loop  ·  {source_path}")

    # ── 1. Generate initial tests ──────────────────────────────────────────────
    print("Step 1: generating tests…")
    test_code = generate_tests(source_path, repo_root=repo_root)
    full_test.write_text(test_code)
    print(f"Written: {test_path}")

    for attempt in range(1, max_retries + 2):   # +1 for initial attempt
        # ── 2. Run tests ──────────────────────────────────────────────────────
        show_rule(f"Attempt {attempt}")
        result = run_tests(test_path, repo_root=repo_root)

        color = "green" if result["ok"] else "red"
        icon = "✓" if result["ok"] else "✗"
        print(f"  {icon} passed={result['passed']}  failed={result['failed']}  errors={result['errors']}")

        if result["ok"]:
            print("\n✓ All tests pass.")
            result["attempts"] = attempt
            return result

        if attempt > max_retries:
            print(f"\nMax retries ({max_retries}) reached. Giving up.")
            break

        # ── 3. Fix ────────────────────────────────────────────────────────────
        print("\nTests failed. Asking model to fix…")
        print(result['output'][-1200:])   # last 1200 chars of pytest output

        fix_instruction = (
            f"The test file has failures. Fix ONLY the test code — do not modify the "
            f"source module. Here is the pytest output:\n\n{result['output'][-1500:]}"
        )
        _, fixed_code = apply_patch(test_path, fix_instruction, repo_root=repo_root)
        full_test.write_text(fixed_code)
        print(f"Rewritten: {test_path}")

    result["attempts"] = max_retries + 1
    return result

### 13.4 Try it — generate and run tests for `formatter.py`

This cell writes real files to `sample_project/tests/` and runs `pytest`.
Make sure `pytest` is installed (`pip install pytest`).

In [ ]:
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "pytest"],
    stdout=subprocess.DEVNULL,
)

result = test_loop(
    source_path = "utils/formatter.py",
    repo_root   = "sample_project",
    max_retries = 3,
)

icon = "✓ PASS" if result["ok"] else "✗ FAIL"
print(f"\nFinal result: {icon}  in {result['attempts']} attempt(s)")

---

## Chapter 14 — Adding a New Capability

The agent is a pipeline of functions connected by a small number of **switch points** — places where the code checks an action type or strategy name and branches. Adding a new capability always means the same three things:

1. **Write the function** — pure Python, no magic.
2. **Wire it in** — add a case to the right switch point.
3. **Tell the planner** — add a line to `_PLAN_SYSTEM` so the LLM knows the action exists.

That's the whole pattern. Everything downstream — the agent loop, the Streamlit UI, the activity log — picks it up automatically.

### The three switch points

| What you're adding | Switch point |
|---|---|
| New retrieval strategy | `pick_strategy()` + new `*_retrieve()` function |
| New plan action | `_PLAN_SYSTEM` prompt + `plan_task()` parser + `execute_step()` |
| New file types to index/embed | `CODE_EXTENSIONS` set |

### 14.1 The worked example: `ACTION: fetch`

We'll add the ability for the agent to fetch a URL — useful when the task says "implement this API" and you paste in a link to the docs.

The plan action will look like:

```
STEP: Read the requests library docs
ACTION: fetch
URL: https://docs.python-requests.org/en/latest/
```

Three changes required. We'll do them one at a time.


### 14.1 — Write the function

`fetch_url()` downloads a page and strips HTML tags so the model gets readable text rather than a wall of `<div>` soup. Nothing agent-specific here — just plain Python.


In [ ]:
import re as _re

def fetch_url(url: str, max_chars: int = 8000) -> dict:
    """
    Fetch *url* and return plain text (HTML tags stripped).

    Returns {"url": str, "text": str, "ok": bool, "error": str | None}

    max_chars caps the text fed to the model — full pages can be huge.
    """
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent": "pocket-agent/1.0"})
        resp.raise_for_status()
        # Strip every HTML tag with a simple regex — good enough for docs pages
        text = _re.sub(r"<[^>]+>", " ", resp.text)
        # Collapse whitespace
        text = _re.sub(r"\s{2,}", "\n", text).strip()
        return {"url": url, "text": text[:max_chars], "ok": True, "error": None}
    except Exception as exc:
        return {"url": url, "text": "", "ok": False, "error": str(exc)}

# Quick smoke-test
result = fetch_url("https://httpbin.org/html")
print(f"ok={result['ok']}  chars={len(result['text'])}")
print(result["text"][:300])


### 14.2 — Wire it into `execute_step()`

`execute_step()` is the second switch point. It already handles `read`, `write`, and `bash`. We add a `fetch` branch that calls `fetch_url()` and returns the same structured dict shape the rest of the loop expects.

Note that the fetched text is also passed to the LLM as context — so the agent can answer questions *about* the page, not just confirm it was downloaded.


In [ ]:
# We redefine execute_step with the fetch branch added.
# Every other branch is identical to Chapter 12.

def execute_step(
    step:      dict,
    repo_root: str = REPO_ROOT,
) -> dict:
    """
    Execute one plan step.  Returns a structured result dict.

      read  → {"type": "read",  "output": str}
      write → {"type": "write", "file_path": str, "new_content": str, "diff": str}
      bash  → {"type": "bash",  "cmd": str, "output": str, "ok": bool}
      fetch → {"type": "fetch", "url": str, "output": str, "ok": bool}   ← NEW
    """
    action = step.get("action", "read")

    # ── NEW: fetch a URL and surface the text ────────────────────────────────
    if action == "fetch":
        url    = step.get("url", step.get("target", ""))
        result = fetch_url(url)
        if not result["ok"]:
            return {"type": "fetch", "url": url, "output": f"Error: {result['error']}", "ok": False}

        # Pass the page text to the LLM so the agent can reason about it
        page_text = result["text"]
        summary_reply, _ = chat([
            {"role": "system", "content": "Summarise the following web page concisely."},
            {"role": "user",   "content": f"URL: {url}\n\n{page_text}"},
        ])
        return {"type": "fetch", "url": url, "output": summary_reply, "ok": True}

    # ── bash ─────────────────────────────────────────────────────────────────
    if action == "bash":
        import subprocess
        cmd  = step.get("cmd", "")
        proc = subprocess.run(cmd, shell=True, cwd=repo_root, capture_output=True, text=True)
        return {"type": "bash", "cmd": cmd,
                "output": (proc.stdout + proc.stderr).strip(),
                "ok": proc.returncode == 0}

    # ── write ─────────────────────────────────────────────────────────────────
    if action == "write":
        original, proposed, diff = write_file(
            file_path=step["target"], instruction=step["instruction"],
            repo_root=repo_root, dry_run=True,
        )
        return {"type": "write", "file_path": step["target"],
                "new_content": proposed, "diff": diff}

    # ── read (default) ────────────────────────────────────────────────────────
    result = run(step["target"], repo_root=repo_root)
    return {"type": "read", "output": result.reply}


### 14.3 — Tell the planner

`_PLAN_SYSTEM` is a plain string that the LLM reads before generating a plan. If `fetch` isn't in it, the model won't emit it — it simply doesn't know the action exists. One new block in the format description and one rule line is all it takes.

We also add `URL:` to the parser's regex so the field is captured correctly.


In [ ]:
_PLAN_SYSTEM = """\
You are a coding agent. Given a task and a project map, produce an ordered
list of ALL steps needed to fully complete the task.

Each step must be one of these four formats (blank line between steps):

STEP: <one-sentence description>
ACTION: read
TARGET: <question to answer about the codebase>

STEP: <one-sentence description>
ACTION: write
TARGET: <file path relative to repo root>
INSTRUCTION: <precise, self-contained instruction for what to write>

STEP: <one-sentence description>
ACTION: bash
CMD: <shell command, e.g. cp foo.md bar.md>

STEP: <one-sentence description>
ACTION: fetch
URL: <full URL to retrieve>

Rules:
- Prefer ACTION: bash for file system operations (copy, move, delete, rename).
- Use ACTION: fetch when the task references an external URL or documentation link.
- Use ACTION: read only when you need code understanding before you can write.
- Use ACTION: write when the LLM must generate or edit file content.
- Emit at most 6 steps. Do not add any text outside the step blocks.
"""


def plan_task(task: str, repo_root: str = REPO_ROOT) -> list[dict]:
    """
    Ask the LLM to produce a step-by-step plan for *task*.
    Returns a list of {"step", "action", "target", "instruction", "cmd", "url"} dicts.
    """
    manifest = load_manifest(repo_root)
    prompt   = f"PROJECT MAP:\n{manifest['text']}\n\nTASK: {task}"
    raw, _   = chat([
        {"role": "system", "content": _PLAN_SYSTEM},
        {"role": "user",   "content": prompt},
    ])
    steps = []
    for block in re.split(r"\n{2,}", raw.strip()):
        lines = {
            m.group(1).lower(): m.group(2).strip()
            for line in block.splitlines()
            if (m := re.match(
                r"^(STEP|ACTION|TARGET|INSTRUCTION|CMD|URL):\s*(.+)$",
                line, re.IGNORECASE
            ))
        }
        action = lines.get("action", "").lower()
        has_content = (
            "target" in lines   # read / write
            or "cmd"  in lines  # bash
            or "url"  in lines  # fetch  ← NEW
            or action in ("bash", "fetch")
        )
        if action and has_content:
            steps.append({
                "step":        lines.get("step", ""),
                "action":      action,
                "target":      lines.get("target", ""),
                "instruction": lines.get("instruction", ""),
                "cmd":         lines.get("cmd", ""),
                "url":         lines.get("url", ""),  # ← NEW
            })
    return steps


### 14.4 Try it — run the agent with a URL task

The cell below asks the agent to fetch the Python `pathlib` docs and write a one-page cheat sheet based on them. The plan should come back with a `fetch` step followed by a `write` step — no reads needed.


In [ ]:
task = (
    "Fetch https://docs.python.org/3/library/pathlib.html "
    "and write a concise cheat sheet to docs/pathlib_cheatsheet.md"
)

plan = plan_task(task, repo_root=REPO_ROOT)

show_rule("Plan")
for i, s in enumerate(plan, 1):
    color = {"fetch": "yellow", "write": "green", "bash": "cyan"}.get(s["action"], "white")
    print(f"{i}. [{color}]{s['action'].upper()}[/{color}]  {s['step']}")
    if s["url"]:
        print(f"   url:   {s['url']}")
    if s["target"]:
        print(f"   target: {s['target']}")
    if s["cmd"]:
        print(f"   cmd:   {s['cmd']}")


Now execute the plan step by step. Accept the write when the diff looks good.


In [ ]:
import os

os.makedirs("docs", exist_ok=True)

for i, step in enumerate(plan, 1):
    show_rule(f"Step {i}/{len(plan)}  {step['action'].upper()}  {step['step']}")

    result = execute_step(step, repo_root=REPO_ROOT)

    if result["type"] == "fetch":
        icon = "✅" if result["ok"] else "❌"
        print(f"{icon} fetched {result['url']}")
        print(f"\nSummary:")
        show_reply(result["output"])

    elif result["type"] == "write":
        print(f"Proposed file: {result['file_path']}")
        print(result["diff"][:1200])
        answer = input("\nAccept this write? [y/N] ").strip().lower()
        if answer == "y":
            path = Path(REPO_ROOT) / result["file_path"]
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(result["new_content"])
            print(f"Written → {result['file_path']}")
        else:
            print("Skipped.")

    elif result["type"] == "bash":
        icon = "✅" if result["ok"] else "❌"
        print(f"{icon} `{result['cmd']}`")
        if result["output"]:
            print(f"{result['output'][:400]}")

    else:  # read
        show_reply(result["output"])


### 14.5 Your turn — exercises

**Exercise A — add a new plan action: `ACTION: notify`**

Wire a new action that sends a desktop notification when the agent finishes a task.
Use Python's `subprocess` to call `osascript` (macOS) or `notify-send` (Linux).

Three things to add:
1. A `notify()` function that runs the OS command
2. A branch in `execute_step()` for `action == "notify"`
3. A block in `_PLAN_SYSTEM` describing the format

The Streamlit UI will pick it up automatically — no UI changes needed.


In [ ]:
# Exercise A — ACTION: notify
#
# Step 1: write the function
import platform, subprocess

def notify(message: str, title: str = "Pocket Agent") -> dict:
    """Send a desktop notification. Returns {"ok": bool, "error": str|None}"""
    try:
        if platform.system() == "Darwin":
            subprocess.run(
                ["osascript", "-e", f'display notification "{message}" with title "{title}"'],
                check=True
            )
        else:
            subprocess.run(["notify-send", title, message], check=True)
        return {"ok": True, "error": None}
    except Exception as e:
        return {"ok": False, "error": str(e)}

# Step 2: add to execute_step() — insert before the final "read" default:
#   if action == "notify":
#       msg = step.get("target", step.get("cmd", "Task complete"))
#       result = notify(msg)
#       return {"type": "notify", "output": msg, "ok": result["ok"]}

# Step 3: add to _PLAN_SYSTEM:
#   STEP: <description>
#   ACTION: notify
#   TARGET: <message to show>

# Test it:
result = notify("Agent finished!", title="Pocket Agent")
print("Notification sent:", result)


**Exercise B — add a new retrieval strategy: recency**

Some tasks are best answered by the files changed most recently — e.g. "what did I just break?". Add a `recent` strategy that ranks files by modification time instead of relevance score.


In [ ]:
# Exercise B — recency retrieval strategy
#
# Step 1: write the function
def recent_retrieve(repo_root: str = REPO_ROOT, top_k: int = 8) -> list[dict]:
    """Return the top-k most recently modified source files."""
    files = glob_files("*", repo_root=repo_root)
    files = [f for f in files if Path(f["path"]).suffix.lower() in CODE_EXTENSIONS]
    # TODO: sort by modification time and return top_k
    # Hint: (Path(repo_root) / f["path"]).stat().st_mtime
    raise NotImplementedError("complete this function")


# Step 2: wire into pick_strategy()
# Add a detection rule — if the query mentions words like "recent", "last",
# "changed", "broke", "latest", route to "recent".
#
# In pick_strategy(), add:
#   if any(w in q for w in ("recent", "last", "changed", "broke", "latest")):
#       return "recent"
#
# Then in run(), add a branch:
#   elif strat == "recent":
#       candidates = recent_retrieve(repo_root=repo_root, top_k=top_k)


# Step 3: test it
# Uncomment once you've implemented the function and wired it in:
# result = run("what files changed most recently?", repo_root=REPO_ROOT)
# show_reply(result.reply)
# print("Strategy:", result.strategy)


**Exercise C — add a new file type: `.sql`**

The agent currently ignores `.sql` files. One line change to `CODE_EXTENSIONS` fixes that — but you also need to clear the embed cache so the index is rebuilt with the new files included.


In [ ]:
# Exercise C — index .sql files

# Step 1: add the extension
CODE_EXTENSIONS.add(".sql")
print("Extensions now include:", sorted(CODE_EXTENSIONS))

# Step 2: the embed index is cached per repo_root — clear it so the
# next semantic_retrieve() call picks up any .sql files it finds.
_EMBED_INDEX.clear()
print("Embed index cleared — will rebuild on next semantic query.")

# Step 3: create a tiny SQL file and verify it gets indexed
import os
os.makedirs("sample_project", exist_ok=True)
Path("sample_project/schema.sql").write_text(
    "CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, email TEXT);\n"
    "CREATE TABLE orders (id INTEGER PRIMARY KEY, user_id INTEGER, total REAL);\n"
)

idx = build_index(repo_root="sample_project")
sql_entries = [e for e in idx if e["path"].endswith(".sql")]
print(f"\nSQL files in index: {[e['path'] for e in sql_entries]}")


---

## Chapter 15 — Streamlit App

The `agent/` directory contains two ready-to-run files built from everything in this notebook:

- **`agent/core.py`** — the full pipeline (constants, retrieval, planning, execution)
- **`agent/app.py`** — the Streamlit UI (Ask tab for streaming Q&A, Agent tab for plan + execute)

This chapter installs Streamlit and launches the app. No re-explanation of the pipeline needed — you already built it.


### 15.1 Install Streamlit


In [7]:
import subprocess, sys
from pathlib import Path

# Install Streamlit if not already present
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "streamlit"],
    stdout=subprocess.DEVNULL,
)

# Create the agent package directory
Path("agent").mkdir(exist_ok=True)
Path("agent/__init__.py").touch()   # makes agent/ a proper Python package

print("Streamlit installed. agent/ directory ready.")

Streamlit installed. agent/ directory ready.


### 15.2 Launch

Run the cell below once. It opens the app in your browser at `http://localhost:8501`.
The Streamlit process runs until you interrupt the kernel cell (`■ Stop`).


In [ ]:
# Run this cell to launch the UI (opens in your browser at http://localhost:8501)
# The process runs in the background — re-run the cell to restart it.
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "agent/app.py",
     "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(f"Streamlit started (PID {proc.pid}).")
print("Open http://localhost:8501 in your browser.")
print("To stop:  proc.terminate()")


---
## You're done.

You've built a fully functional local coding agent from scratch, one capability at a time.

| Ch | What you built | Key function |
|----|---------------|--------------|
| 1 | LLM connection + status panel | `chat()`, `show_panel()` |
| 2 | Token budget awareness | `count_tokens()`, `scan_repo_costs()` |
| 3 | Project manifest | `load_manifest()`, `ask_with_manifest()` |
| 4 | File navigation without bulk loading | `glob_files()`, `jit_read()`, `budget_load()` |
| 5 | Relevance ranking by filename | `score_file()`, `rank_files()` |
| 6 | Content-based retrieval | `grep_repo()`, `grep_rank()` |
| 7 | Long-session survival | `compact()`, `summarise_file()` |
| 8 | Meaning-based retrieval | `embed()`, `semantic_retrieve()` |
| 9 | Unified pipeline | `run()`, `pick_strategy()` |
| 10 | File modification with diff preview | `write_file()`, `make_diff()` |
| 11 | Autonomous task execution | `agent_loop()`, `plan_task()` |
| 12 | Self-verifying test generation | `test_loop()`, `generate_tests()` |

### What to try next

- Point `REPO_ROOT` and `run()` at a real project you're working on
- Swap `OLLAMA_MODEL` for a larger model (`qwen4.5:14b`, `devstral-small-2`) and compare output quality
- Extend `plan_task()` to support a `refactor` action type
- Persist the embedding index to disk so it survives notebook restarts
- Add a `AGENTS.md` to your own repos